# WtCore 模块架构


```mermaid
graph TB
    %% 样式定义
    classDef engineClass fill:#e1f5ff,stroke:#01579b,stroke-width:2px
    classDef contextClass fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef adapterClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef dataClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px
    classDef executerClass fill:#fff9c4,stroke:#f57f17,stroke-width:2px
    classDef utilClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px
    classDef tickerClass fill:#e0f2f1,stroke:#004d40,stroke-width:2px
    classDef mgrClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px
    
    %% 接口层（简化表示）
    subgraph Interfaces["接口层"]
        I3["ITrdNotifySink<br/>交易通知接口"]
        I4["IExecCommand<br/>执行命令接口"]
    end
    
    %% 引擎层
    subgraph Engines["引擎层"]
        WtEngine["WtEngine<br/>引擎基类"]
        WtCtaEngine["WtCtaEngine<br/>CTA引擎"]
        WtHftEngine["WtHftEngine<br/>高频引擎"]
        WtSelEngine["WtSelEngine<br/>选股引擎"]
    end
    
    %% 策略管理器层
    subgraph StrategyMgrs["策略管理器层"]
        StrategyMgr["{Cta/Hft/Sel}StrategyMgr<br/>策略管理器<br/>创建策略实例"]
    end
    
    %% Ticker层
    subgraph Tickers["Ticker层"]
        WtTicker["Wt{Cta/Hft/Sel}Ticker<br/>CTA/高频/选股实时时钟"]
    end
    
    %% 策略上下文层
    subgraph Contexts["策略上下文层"]
        StraBaseCtx["{Cta/Hft/Sel}StraBaseCtx<br/>策略基础上下文<br/>实现核心功能"]
        StraContext["{Cta/Hft/Sel}StraContext<br/>策略上下文<br/>策略回调转发"]
    end
    
    %% 适配器层
    subgraph Adapters["适配器层"]
        TraderAdapter["TraderAdapter<br/>交易适配器"]
        ParserAdapter["ParserAdapter<br/>解析器适配器"]
    end
    
    %% 数据管理层
    subgraph DataLayer["数据管理层"]
        WtDtMgr["WtDtMgr<br/>数据管理器"]
    end
    
    %% 执行器层
    subgraph ExecuterLayer["执行器层"]
        WtExecMgr["WtExecMgr<br/>执行器管理器"]
        WtExecuterFactory["WtExecuterFactory<br/>执行器工厂<br/>创建执行单元"]
        ExecuterImpl["执行器<br/>Wt{Local/Arbi/Diff/Dist}Executer"]
    end
    
    %% 工具层
    subgraph Utils["工具支持层"]
        EventNotifier["EventNotifier<br/>事件通知器"]
        WtFilterMgr["WtFilterMgr<br/>过滤器管理器"]
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>开平仓策略"]
        WtHelper["WtHelper<br/>辅助工具"]
    end
    
    %% 继承关系（简化）
    WtCtaEngine -.->|继承| WtEngine
    WtHftEngine -.->|继承| WtEngine
    WtSelEngine -.->|继承| WtEngine
    
    StraContext -.->|继承| StraBaseCtx
    StraBaseCtx -.->|实现| I3
    
    ExecuterImpl -.->|实现| I4
    
    %% 核心组合关系
    WtCtaEngine -->|管理| StraContext
    WtHftEngine -->|管理| StraContext
    WtSelEngine -->|管理| StraContext
    
    WtCtaEngine -->|包含| WtTicker
    WtHftEngine -->|包含| WtTicker
    WtSelEngine -->|包含| WtTicker
    
    WtCtaEngine -->|使用| WtExecMgr
    WtSelEngine -->|使用| WtExecMgr
    WtExecMgr -->|管理| ExecuterImpl
    
    StrategyMgr -->|创建| StraBaseCtx
    
    ExecuterImpl -->|使用| WtExecuterFactory
    
    %% 依赖关系
    WtEngine -->|使用| WtDtMgr
    WtEngine -->|使用| TraderAdapter
    WtEngine -->|使用| EventNotifier
    
    WtCtaEngine -->|使用| WtDtMgr
    WtHftEngine -->|使用| WtDtMgr
    WtSelEngine -->|使用| WtDtMgr
    
    ExecuterImpl -->|使用| TraderAdapter
    
    TraderAdapter -->|使用| EventNotifier
    TraderAdapter -->|使用| ActionPolicyMgr
    
    %% 应用样式
    class WtEngine,WtCtaEngine,WtHftEngine,WtSelEngine engineClass
    class StraBaseCtx,StraContext contextClass
    class TraderAdapter,ParserAdapter adapterClass
    class WtDtMgr dataClass
    class WtExecMgr,WtExecuterFactory,ExecuterImpl executerClass
    class EventNotifier,WtFilterMgr,ActionPolicyMgr,WtHelper utilClass
    class WtTicker tickerClass
    class StrategyMgr mgrClass
```


# 工具支持层

## 事件通知器 EventNotifier.h/cpp
用于将交易系统中的各种事件通过消息队列（MQ）进行异步广播。
```
主线程（交易线程）             工作线程 _worker（异步IO线程）
     |                              |
     | notify_*() 调用              |
     |      |                       |
     |      v                       |
     |  _asyncio.post()             |
     |      |                       |
     |      | 投递任务到队列         |
     |      |                       |
     |      |---------------------->|
     |      |                       |  _asyncio.run_one()
     |      |                       |      |
     |      |                       |      v
     |      |                       |  执行任务
     |      |                       |  (JSON转换、发布消息)
     |      |                       |
     |  立即返回                     |
     |  (继续处理交易)               |
```

**成员**：
- `std::string _url`：消息队列服务器地址/URL
- `uint32_t _mq_sid`：消息队列服务器ID
- `FuncCreateMQServer _creator`：创建MQ服务器的函数指针
    ```cpp
    /* @param 消息队列服务器地址/URL
     * @return 返回MQ服务器ID（无符号长整型）*/
    typedef unsigned long(*FuncCreateMQServer)(const char*);
    ```
- `FuncDestroyMQServer _remover`：销毁MQ服务器的函数指针
    ```cpp
    /* @param mq_id MQ服务器ID */
    typedef void(*FuncDestroyMQServer)(unsigned long);
    ```
- `FundPublishMessage _publisher`：发布消息到消息队列的函数指针
    ```cpp
    /* @param mq_id MQ服务器ID
     * @param topic 消息主题
     * @param data 消息数据内容
     * @param length 消息数据长度
     */
    typedef void(*FundPublishMessage)(unsigned long, const char*, const char*, unsigned long);
    ```
- `FuncRegCallbacks	_register`：注册回调函数到消息队列的函数指针
    ```cpp
    /* @param callback 日志回调函数指针 */
    typedef void(*FuncRegCallbacks)(FuncLogCallback);
    ```
- `bool _stopped`：停止标志，用于控制异步IO工作线程退出
- `boost::asio::io_service _asyncio`：异步IO服务对象，用于异步事件处理
- `StdThreadPtr _worker`：异步IO工作线程指针

**方法**：
- **初始化与配置**
  - 初始化事件通知器：`bool init(WTSVariant* cfg)`

- **交易事件通知**
  - 通知交易事件：`void notify(const char* trader, uint32_t localid, const char* stdCode, WTSTradeInfo* trdInfo)`
  - 通知订单事件：`void notify(const char* trader, uint32_t localid, const char* stdCode, WTSOrderInfo* ordInfo)`
  - 通知交易接口消息：`void notify(const char* trader, const char* message)`
  - 通知策略交易事件：`void notify_trade(const char* straId, const char* stdCode, bool isLong, bool isOpen, uint64_t curTime, double price, const char* userTag)`

- **日志事件通知**
  - 通知日志事件：`void notify_log(const char* tag, const char* message)`

- **图表事件通知**
  - 通知图表标记事件：`void notify_chart_marker(uint64_t time, const char* straId, double price, const char* icon, const char* tag)`
  - 通知图表指标事件：`void notify_chart_index(uint64_t time, const char* straId, const char* idxName, const char* lineName, double val)`

- **通用事件通知**
  - 通知通用事件：`void notify_event(const char* message)`

- **私有辅助方法**
  - 交易信息转JSON：`void tradeToJson(const char* trader, uint32_t localid, const char* stdCode, WTSTradeInfo* trdInfo, std::string& output)`
  - 订单信息转JSON：`void orderToJson(const char* trader, uint32_t localid, const char* stdCode, WTSOrderInfo* ordInfo, std::string& output)`

- **初始化与配置**
  - 初始化事件通知器：`bool init(WTSVariant* cfg)`

- **交易事件通知**
  - 通知交易事件：`void notify(const char* trader, uint32_t localid, const char* stdCode, WTSTradeInfo* trdInfo)`
  - 通知订单事件：`void notify(const char* trader, uint32_t localid, const char* stdCode, WTSOrderInfo* ordInfo)`
  - 通知交易接口消息：`void notify(const char* trader, const char* message)`
  - 通知策略交易事件：`void notify_trade(const char* straId, const char* stdCode, bool isLong, bool isOpen, uint64_t curTime, double price, const char* userTag)`

- **日志事件通知**
  - 通知日志事件：`void notify_log(const char* tag, const char* message)`

- **图表事件通知**
  - 通知图表标记事件：`void notify_chart_marker(uint64_t time, const char* straId, double price, const char* icon, const char* tag)`
  - 通知图表指标事件：`void notify_chart_index(uint64_t time, const char* straId, const char* idxName, const char* lineName, double val)`

- **通用事件通知**
  - 通知通用事件：`void notify_event(const char* message)`

- **私有辅助方法**
  - 交易信息转JSON：`void tradeToJson(const char* trader, uint32_t localid, const char* stdCode, WTSTradeInfo* trdInfo, std::string& output)`
  - 订单信息转JSON：`void orderToJson(const char* trader, uint32_t localid, const char* stdCode, WTSOrderInfo* ordInfo, std::string& output)`

## 动作策略管理器 ActionPolicyMgr.h/cpp

### 交易动作类型 ActionType
```cpp
/**
 * @enum ActionType
 * @brief 交易动作类型枚举
 * 
 * 定义交易中可能执行的动作类型，包括开仓、平仓、平今、平昨等操作。
 * 用于标识交易指令的具体动作类型。
 */
typedef enum tagActionType
{
	AT_Unknown = 8888, // 未知动作类型，默认值
	AT_Open = 9999, // 开仓动作：买入或卖出建立新仓位
	AT_Close, // 平仓动作：关闭现有仓位（不区分今昨）
	AT_CloseToday, // 平今动作：只平今日开仓的仓位
	AT_CloseYestoday // 平昨动作：只平昨日及之前开仓的仓位
} ActionType;
```

### 交易动作规则 ActionRule
```cpp
/**
 * @struct ActionRule
 * @brief 动作规则结构体
 * 
 * 定义单个交易动作的执行规则，包括动作类型、手数限制等约束条件。
 * 用于控制交易动作的执行范围和行为。
 */
typedef struct _ActionRule
{
	ActionType	_atype;	// 动作类型：标识执行的具体动作（开仓、平仓等）
	uint32_t	_limit;	// 手数限制：总体手数限制，0表示无限制
	uint32_t	_limit_l;	// 多头手数限制：限制多头方向的最大手数，0表示无限制
	uint32_t	_limit_s;	// 空头手数限制：限制空头方向的最大手数，0表示无限制
	bool		_pure;	// 净仓标志：主要针对AT_CloseToday和AT_CloseYestoday，true表示必须是净今仓或净昨仓才能执行
} ActionRule;
```

### 动作策略管理器类 ActionPolicyMgr
管理交易动作的执行规则，通过品种ID映射到对应的规则组，实现不同品种的差异化规则管理。

成员：
- `RulesMap _rules`：规则表，存储所有已加载的规则组，key为规则组名称
  - typedef wt_hashmap\<std::string, `ActionRuleGroup`\> RulesMap;
  - typedef std::vector\<`ActionRule`\>	ActionRuleGroup;
- `wt_hashmap<std::string, std::string> _comm_rule_map`：品种规则映射，品种ID -> 规则组名称

## 过滤器管理器 WtFilterMgr.h/cpp
用于管理和执行交易信号的过滤规则

**成员**：
- `FilterMap _stra_filters`：策略过滤器映射表，策略名称 ——> 过滤器
- `FilterMap _code_filters`：代码过滤器映射表，合约代码或品种代码 ——> 过滤器
  - typedef wt_hashmap\<std::string, `FilterItem`\>	FilterMap：过滤器映射表，过滤器ID ——> 过滤器
    ```cpp
    /* 存储单个过滤器的配置 */
    typedef struct _FilterItem
    {
      std::string _key; // 过滤器关键字，用于匹配策略名称或合约代码
      FilterAction _action; // 过滤操作类型，决定是忽略还是重定向
      double _target; // 目标仓位值，只有当_action为FA_Redirect时才生效
    } FilterItem;

    /* 过滤器操作类型 */
    typedef enum tagFilterAction
    {
      FA_Ignore,  // 忽略操作，即维持原有仓位，不执行该信号
      FA_Redirect,  // 重定向持仓操作，即同步到指定目标仓位
      FA_None = 99  // 无操作，表示未定义的操作类型
    } FilterAction;
    ```
- `ExecuterFilters _exec_filters`：存储所有被禁用的过滤器ID
  - typedef wt_hashmap\<std::string, bool\> ExecuterFilters：过滤器ID ——> 是否禁用
- `std::string _filter_file`：过滤器配置文件路径
- `uint64_t _filter_timestamp`：过滤器文件最后修改时间戳，用于检测文件是否被修改
- `EventNotifier* _notifier`：事件通知器指针，用于通知过滤器配置变化

**方法**：
- 设置事件通知器：`void set_notifier(EventNotifier* notifier)`
- 加载信号过滤器：`void load_filters(const char* fileName = "")`
- 检查策略是否被过滤：`bool is_filtered_by_strategy(const char* straName, double& targetPos, bool isDiff = false)`
- 检查合约是否被过滤：`bool is_filtered_by_code(const char* stdCode, double& targetPos)`
- 检查执行器是否被过滤：`bool is_filtered_by_executer(const char* execid)`

## 辅助工具类 WtHelper.h/cpp
提供系统路径管理和时间管理功能
- 当前工作目录 (程序运行时的当前工作目录)
- 实例目录：`_inst_dir`，用于区分不同实例
- 生成文件输出目录：`_gen_dir`，默认为 "./generated/"
  - 输出目录：_gen_dir + "outputs/"
  - 策略数据目录：_gen_dir + "stradata/"
  - 策略用户数据目录：_gen_dir + "userdata/"
  - 组合目录：_gen_dir + "portfolio/"
  - 执行数据目录：_gen_dir + "execdata/"

**成员**：
- `static uint32_t _cur_date`：当前日期，格式为YYYYMMDD
- `static uint32_t _cur_time`：当前时间（分钟），格式为HHMM
- `static uint32_t _cur_secs`：当前秒数，包含毫秒信息
- `static uint32_t _cur_tdate`：当前交易日，格式为YYYYMMDD
- `static std::string _inst_dir`：实例所在目录路径
- `static std::string _gen_dir`：生成文件输出目录路径，默认为"./generated/"

**方法**：
- **路径管理方法**
  - 获取当前工作目录：`static std::string getCWD()`
  - 获取模块路径：`static std::string getModulePath(const char* moduleName, const char* subDir, bool isCWD = true)`
  - 获取基础目录：`static const char* getBaseDir()`
  - 获取输出目录：`static const char* getOutputDir()`
  - 获取策略数据目录：`static const char* getStraDataDir()`
  - 获取策略用户数据目录：`static const char* getStraUsrDatDir()`
  - 获取组合目录：`static const char* getPortifolioDir()`
  - 获取执行数据目录：`static const char* getExecDataDir()`
  - 获取实例目录：`static const std::string& getInstDir()`

- **路径设置方法**
  - 设置实例目录：`static void setInstDir(const char* inst_dir)`
  - 设置生成文件输出目录：`static void setGenerateDir(const char* gen_dir)`

- **时间设置方法**
  - 设置当前时间：`static void setTime(uint32_t date, uint32_t time, uint32_t secs = 0)`
  - 设置交易日期：`static void setTDate(uint32_t tDate)`

- **时间获取方法**
  - 获取当前日期：`static uint32_t getDate()`
  - 获取当前时间：`static uint32_t getTime()`
  - 获取当前秒数：`static uint32_t getSecs()`
  - 获取交易日期：`static uint32_t getTradingDate()`

# 接口层

## 交易通知接收器接口 ITrdNotifySink.h
用于接收交易相关的各种通知和回调。采用 ***观察者模式***：
- 一个 ITrdNotifySink 实例作为一个**观察者**，适配器层/TraderAdapter 作为**被观察者**
- 被观察者包含多个观察者，一旦被观察者状态发生变化，其会通知所有观察值响应

方法：
- **交易事件回调接口**
  - **成交回报回调**：`virtual void on_trade(uint32_t localid, const char* stdCode, bool isBuy, double vol, double price) = 0;`
    - 当订单成交时被调用，通知接收器成交信息（纯虚函数，必须实现）
  - **订单回报回调**：`virtual void on_order(uint32_t localid, const char* stdCode, bool isBuy, double totalQty, double leftQty, double price, bool isCanceled = false) = 0;`
    - 当订单状态发生变化时被调用，通知接收器订单状态信息（纯虚函数，必须实现）
  - **持仓更新回调**：`virtual void on_position(const char* stdCode, bool isLong, double prevol, double preavail, double newvol, double newavail, uint32_t tradingday) {}`
    - 当持仓发生变化时被调用，通知接收器持仓变化信息（默认实现为空，可选择性实现）
  - **下单回报回调**：`virtual void on_entrust(uint32_t localid, const char* stdCode, bool bSuccess, const char* message) {}`
    - 当下单结果返回时被调用，通知接收器下单是否成功（默认实现为空，可选择性实现）
  - **资金回调**：`virtual void on_account(const char* currency, double prebalance, double balance, double dynbalance, double avaliable, double closeprofit, double dynprofit, double margin, double fee, double deposit, double withdraw) {}`
    - 当账户资金发生变化时被调用，通知接收器资金变化信息（默认实现为空，可选择性实现）
- **通道状态回调接口**
  - **交易通道就绪回调**：`virtual void on_channel_ready() = 0;`
    - 当交易通道就绪时被调用，通知接收器可以开始交易（纯虚函数，必须实现）
  - **交易通道丢失回调**：`virtual void on_channel_lost() = 0;`
    - 当交易通道丢失时被调用，通知接收器交易通道已断开（纯虚函数，必须实现）

## 执行命令接口 IExecCommand.h
```mermaid
graph TB

    %% 样式定义
    classDef interfaceClass fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    classDef commandClass fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px
    classDef stubClass fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef executerClass fill:#e1f5ff,stroke:#01579b,stroke-width:2px
    classDef dataClass fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px

    %% 接口层
    subgraph InterfaceLayer["接口层"]
        IExecuterStub["IExecuterStub<br/>执行器存根接口<br/>提供执行器所需的基础信息查询<br/>• 获取实时时间<br/>• 获取商品信息<br/>• 获取交易会话信息<br/>• 获取热点合约管理器<br/>• 获取交易日期"]
        
        IExecCommand["IExecCommand<br/>执行命令接口<br/>定义执行器需要实现的命令<br/>• 设置目标仓位<br/>• 仓位变动通知<br/>• 实时行情回调<br/>• 持有IExecuterStub指针"]
    end

    %% 实现层
    subgraph ImplementationLayer["实现层"]
        WtLocalExecuter["WtLocalExecuter<br/>本地执行器<br/>实现IExecCommand<br/>• 管理执行单元<br/>• 处理目标仓位<br/>• 提供交易接口<br/>• 处理交易回报"]
        
        WtArbiExecuter["WtArbiExecuter<br/>套利执行器<br/>实现IExecCommand<br/>• 管理套利策略执行单元<br/>• 处理合约组合<br/>• 自动清理头寸"]
        
        WtDiffExecuter["WtDiffExecuter<br/>差分执行器<br/>实现IExecCommand<br/>• 处理差分交易逻辑"]
        
        WtDistExecuter["WtDistExecuter<br/>分布式执行器<br/>实现IExecCommand<br/>• 处理分布式执行逻辑"]
    end

    %% 存根实现层
    subgraph StubLayer["存根实现层"]
        WtExecRunner["WtExecRunner<br/>执行器运行器<br/>实现IExecuterStub<br/>• 提供执行上下文信息<br/>• 管理执行器生命周期<br/>• 连接交易和解析适配器"]
    end

    %% 依赖组件层
    subgraph DependencyLayer["依赖组件层"]
        WTSCommodityInfo["WTSCommodityInfo<br/>商品信息<br/>• 交易规则<br/>• 手续费信息"]
        
        WTSSessionInfo["WTSSessionInfo<br/>交易会话信息<br/>• 交易时间段<br/>• 开盘时间"]
        
        IHotMgr["IHotMgr<br/>热点合约管理器<br/>• 查询主力合约<br/>• 品种管理"]
        
        WTSTickData["WTSTickData<br/>Tick数据<br/>• 实时行情信息"]
    end

    %% 继承关系
    WtLocalExecuter -.->|实现| IExecCommand
    WtArbiExecuter -.->|实现| IExecCommand
    WtDiffExecuter -.->|实现| IExecCommand
    WtDistExecuter -.->|实现| IExecCommand
    
    WtExecRunner -.->|实现| IExecuterStub

    %% 组合关系
    IExecCommand -->|使用_stub指针| IExecuterStub

    %% 依赖关系
    IExecuterStub -->|获取| WTSCommodityInfo
    IExecuterStub -->|获取| WTSSessionInfo
    IExecuterStub -->|获取| IHotMgr
    IExecCommand -->|接收| WTSTickData

    %% 运行时关系
    WtExecRunner -.->|setStub设置存根| WtLocalExecuter
    WtExecRunner -.->|setStub设置存根| WtArbiExecuter

    %% 应用样式
    class IExecuterStub,IExecCommand interfaceClass
    class WtLocalExecuter,WtArbiExecuter,WtDiffExecuter,WtDistExecuter executerClass
    class WtExecRunner stubClass
    class WTSCommodityInfo,WTSSessionInfo,IHotMgr,WTSTickData dataClass
```

### 执行器存根接口 IExecuterStub
定义了执行器所需的基础信息查询功能：

- **获取实时时间**：`virtual uint64_t get_real_time() = 0;`
  - 返回值：当前实时时间戳（纳秒级）
  - 说明：获取执行器当前的实时时间戳（纯虚函数，必须实现）

- **获取商品信息**：`virtual WTSCommodityInfo* get_comm_info(const char* stdCode) = 0;`
  - 参数：`stdCode`（标准合约代码）
  - 返回值：商品信息对象指针
  - 说明：根据标准合约代码获取对应的商品信息，包括交易规则、手续费等（纯虚函数，必须实现）

- **获取交易会话信息**：`virtual WTSSessionInfo* get_sess_info(const char* stdCode) = 0;`
  - 参数：`stdCode`（标准合约代码）
  - 返回值：交易会话信息对象指针
  - 说明：根据标准合约代码获取对应的交易会话信息，包括交易时间段、开盘时间等（纯虚函数，必须实现）

- **获取热点合约管理器**：`virtual IHotMgr* get_hot_mon() = 0;`
  - 返回值：热点合约管理器接口指针
  - 说明：获取热点合约管理器，用于查询主力合约、次主力合约等（纯虚函数，必须实现）

- **获取交易日期**：`virtual uint32_t get_trading_day() = 0;`
  - 返回值：当前交易日期（格式：YYYYMMDD）
  - 说明：获取执行器当前的交易日期（纯虚函数，必须实现）

### 执行命令接口 IExecCommand
定义了执行器需要实现的命令接口
- 执行命令对象通过实现这些接口，定义具体的执行逻辑。

成员：
- `IExecuterStub* _stub`：执行器存根指针，用于获取执行所需的信息
- `std::string _name`：执行命令名称

方法：
- **设置目标仓位**：`virtual void set_position(const wt_hashmap<std::string, double>& targets) {}`
  - 参数：`targets`（目标仓位映射表，键为合约代码，值为目标持仓数量）
  - 说明：根据目标仓位映射表设置各合约的目标持仓，执行器会根据当前持仓和目标持仓的差异生成相应的交易指令（默认实现为空，可选择性实现）

- **合约仓位变动通知**：`virtual void on_position_changed(const char* stdCode, double diffPos) {}`
  - 参数：`stdCode`（标准合约代码），`diffPos`（仓位变动数量，正数表示增加，负数表示减少）
  - 说明：当合约仓位发生变化时被调用，通知执行命令仓位变动情况，执行命令可以根据仓位变动情况更新内部状态或执行其他逻辑（默认实现为空，可选择性实现）

- **实时行情回调**：`virtual void on_tick(const char* stdCode, WTSTickData* newTick) {}`
  - 参数：`stdCode`（标准合约代码），`newTick`（新的Tick数据指针）
  - 说明：当收到实时行情数据时被调用，通知执行命令最新的行情信息，执行命令可以根据行情数据调整执行策略或触发执行逻辑（默认实现为空，可选择性实现）

# 适配器层

## 解析器适配器 ParserAdapter.h/cpp
```mermaid
graph TB
    %% 样式定义
    classDef interfaceClass fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    classDef adapterClass fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px
    classDef managerClass fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef engineClass fill:#e1f5ff,stroke:#01579b,stroke-width:2px
    classDef dataClass fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    classDef externalClass fill:#fce4ec,stroke:#c2185b,stroke-width:2px
    
    %% 外部接口层
    subgraph ExternalInterfaces["外部接口层"]
        IParserApi["IParserApi<br/>解析器API接口<br/>• 创建解析器<br/>• 订阅合约<br/>• 连接解析器"]
        IParserSpi["IParserSpi<br/>解析器SPI接口<br/>• 接收行情回调<br/>• 接收委托队列<br/>• 接收逐笔数据<br/>• 接收日志"]
    end
    
    %% 内部接口层
    subgraph InternalInterfaces["内部接口层"]
        IParserStub["IParserStub<br/>解析器存根接口<br/>定义引擎接收数据的方法<br/>• handle_push_quote<br/>• handle_push_order_detail<br/>• handle_push_order_queue<br/>• handle_push_transaction"]
    end
    
    %% 适配器层
    subgraph AdapterLayer["适配器层"]
        ParserAdapter["ParserAdapter<br/>解析器适配器<br/>作用：连接解析器和引擎<br/>• 实现IParserSpi接收数据<br/>• 使用IParserStub推送数据<br/>• 数据过滤和转换<br/>• 生命周期管理"]
    end
    
    %% 管理器层
    subgraph ManagerLayer["管理器层"]
        ParserAdapterMgr["ParserAdapterMgr<br/>解析器适配器管理器<br/>作用：管理多个适配器<br/>• 添加/获取适配器<br/>• 批量运行/释放<br/>• 维护适配器映射表"]
    end
    
    %% 引擎层
    subgraph EngineLayer["引擎层"]
        WtEngine["WtEngine<br/>引擎基类<br/>实现IParserStub<br/>接收处理后的行情数据"]
    end
    
    %% 数据管理层
    subgraph DataLayer["数据管理层"]
        IBaseDataMgr["IBaseDataMgr<br/>基础数据管理器<br/>• 查询合约信息<br/>• 获取合约列表<br/>• 代码转换辅助"]
        IHotMgr["IHotMgr<br/>热点合约管理器<br/>• 查询主力合约<br/>• 品种管理"]
    end
    
    %% 继承关系
    ParserAdapter -.->|实现| IParserSpi
    WtEngine -.->|实现| IParserStub
    
    %% 组合关系
    ParserAdapterMgr -->|包含多个| ParserAdapter
    
    %% 依赖关系
    ParserAdapter -->|使用| IParserApi
    ParserAdapter -->|推送数据| IParserStub
    IParserStub -->|被实现| WtEngine
    
    ParserAdapter -->|查询| IBaseDataMgr
    ParserAdapter -->|查询| IHotMgr
    
    ParserAdapterMgr -->|管理| ParserAdapter
    
    %% 数据流
    IParserApi -.->|回调| IParserSpi
    IParserSpi -.->|被实现| ParserAdapter
    ParserAdapter -.->|调用| IParserStub
    
    %% 应用样式
    class IParserApi,IParserSpi,IParserStub interfaceClass
    class ParserAdapter adapterClass
    class ParserAdapterMgr managerClass
    class WtEngine engineClass
    class IBaseDataMgr,IHotMgr dataClass
```

## 交易适配器 TraderAdapter.h/cpp
```mermaid
graph TB

    %% 样式定义
    classDef interfaceClass fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    classDef adapterClass fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px
    classDef managerClass fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef engineClass fill:#e1f5ff,stroke:#01579b,stroke-width:2px
    classDef dataClass fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    classDef notifyClass fill:#fce4ec,stroke:#c2185b,stroke-width:2px
    classDef structClass fill:#fff9c4,stroke:#f57f17,stroke-width:2px

    %% 外部接口层
    subgraph ExternalInterfaces["外部接口层"]
        ITraderApi["ITraderApi<br/>底层交易API接口<br/>• 连接交易服务器<br/>• 登录/注销账户<br/>• 下单/撤单<br/>• 查询账户/持仓/订单"]
        
        ITraderSpi["ITraderSpi<br/>交易回调接口<br/>• 接收登录结果<br/>• 接收订单推送<br/>• 接收成交推送<br/>• 接收查询响应"]
    end

    %% 适配器层
    subgraph AdapterLayer["适配器层"]
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>作用：封装底层交易接口<br/>• 实现ITraderSpi接收回调<br/>• 调用ITraderApi执行交易<br/>• 管理持仓和订单状态<br/>• 实现智能开平仓逻辑<br/>• 风控检查和管理"]
    end

    %% 管理器层
    subgraph ManagerLayer["管理器层"]
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器<br/>作用：管理多个交易通道<br/>• 添加/获取适配器<br/>• 批量启动/释放<br/>• 刷新资金信息<br/>• 维护适配器映射表"]
    end

    %% 依赖组件层
    subgraph DependencyLayer["依赖组件层"]
        IBaseDataMgr["IBaseDataMgr<br/>基础数据管理器<br/>• 查询合约信息<br/>• 获取商品信息<br/>• 代码转换辅助"]
        
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>• 管理开平仓策略规则<br/>• 提供动作规则组<br/>• 策略匹配和选择"]
        
        EventNotifier["EventNotifier<br/>事件通知器<br/>• 发送交易事件通知<br/>• 通知订单/成交变化<br/>• 通知通道状态"]
    end

    %% 通知接口层
    subgraph NotifyLayer["通知接口层"]
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接收器<br/>• 接收成交通知<br/>• 接收订单通知<br/>• 接收持仓更新<br/>• 接收通道状态"]
    end

    %% 内部数据结构层
    subgraph DataStructLayer["内部数据结构层"]
        PosItem["PosItem<br/>持仓项结构<br/>• 多空今昨持仓数据<br/>• 可用持仓计算<br/>• 总持仓计算"]
        
        RiskParams["RiskParams<br/>风控参数结构<br/>• 下单频率限制<br/>• 撤单频率限制<br/>• 时间窗口配置"]
    end

    %% 继承关系
    TraderAdapter -.->|实现| ITraderSpi

    %% 组合关系
    TraderAdapterMgr -->|管理多个| TraderAdapter

    %% 依赖关系
    TraderAdapter -->|调用| ITraderApi
    TraderAdapter -->|查询| IBaseDataMgr
    TraderAdapter -->|使用| ActionPolicyMgr
    TraderAdapter -->|通知| EventNotifier
    TraderAdapter -->|通知集合| ITrdNotifySink
    TraderAdapter -->|包含映射| PosItem
    TraderAdapter -->|包含映射| RiskParams

    %% 回调关系
    ITraderApi -.->|回调| ITraderSpi
    ITraderSpi -.->|被实现| TraderAdapter

    %% 应用样式
    class ITraderApi,ITraderSpi interfaceClass
    class TraderAdapter adapterClass
    class TraderAdapterMgr managerClass
    class IBaseDataMgr,ActionPolicyMgr,EventNotifier dataClass
    class ITrdNotifySink notifyClass
    class PosItem,RiskParams structClass
```

### 交易适配器类 TraderAdapter

成员：
- **配置与标识**
  - `WTSVariant* _cfg`：配置参数对象，保存交易通道的配置信息
  - `std::string _id`：交易通道标识符
  - `std::string _order_pattern`：订单标签模式，用于标识本通道发出的订单，格式为"otp.{通道ID}"
  - `uint32_t _trading_day`：交易日，从登录响应中获取（YYYYMMDD格式）

- **核心接口指针**
  - `ITraderApi* _trader_api`：底层交易API接口指针，提供与交易所交互的底层功能
  - `FuncDeleteTrader _remover`：交易API删除函数指针，用于释放API资源
  - `IBaseDataMgr* _bd_mgr`：基础数据管理器指针，用于获取合约和商品信息
  - `ActionPolicyMgr* _policy_mgr`：动作策略管理器指针，用于管理开平仓策略规则
  - `EventNotifier* _notifier`：事件通知器指针，用于发送交易事件通知

- **状态管理**
  - `AdapterState _state`：当前交易适配器状态枚举
    ```cpp
    /* 定义交易通道从连接、登录到就绪的各个状态 */
    typedef enum tagAdapterState
    {
      AS_NOTLOGIN,  // 未登录状态：初始状态，尚未开始登录流程
      AS_LOGINING,  // 正在登录：已发起登录请求，等待登录结果
      AS_LOGINED, // 已登录：登录成功，但尚未完成数据查询
      AS_LOGINFAILED, // 登录失败：登录请求被拒绝或失败
      AS_POSITION_QRYED,  // 仓位已查：持仓查询完成
      AS_ORDERS_QRYED,  // 订单已查：订单查询完成
      AS_TRADES_QRYED,  // 成交已查：成交查询完成
      AS_ALLREADY // 全部就绪：所有查询完成，交易通道可以使用
    } AdapterState;
    ```
- **通知接收器集合**
  - `wt_hashset<ITrdNotifySink*> _sinks`：交易通知接收器集合，用于通知交易状态变化

- **持仓管理**
  - `wt_hashmap<std::string, PosItem> _positions`：持仓映射表，标准合约代码 ——> 持仓项
    ```cpp
    /* 持仓项结构体，用于存储单个合约的持仓信息 */
    typedef struct _PosItem
    {
      // 多仓数据（做多方向持仓）
      double	l_newvol;		// 多头今仓数量：今日开仓的多头持仓数量
      double	l_newavail;		// 多头今仓可用：今日开仓的多头持仓中可用于平仓的数量
      double	l_prevol;		// 多头昨仓数量：昨日及之前开仓的多头持仓数量
      double	l_preavail;		// 多头昨仓可用：昨日及之前开仓的多头持仓中可用于平仓的数量
      // 空仓数据（做空方向持仓）
      double	s_newvol;		// 空头今仓数量：今日开仓的空头持仓数量
      double	s_newavail;		// 空头今仓可用：今日开仓的空头持仓中可用于平仓的数量
      double	s_prevol;		// 空头昨仓数量：昨日及之前开仓的空头持仓数量
      double	s_preavail;		// 空头昨仓可用：昨日及之前开仓的空头持仓中可用于平仓的数量
    } PosItem;
    ```

- **订单管理**
  - `SpinMutex _mtx_orders`：订单映射表的自旋锁，保证线程安全
  - `OrderMap* _orders`：订单映射表，本地订单ID ——> 订单信息 WTSOrderInfo*
  - `wt_hashset<std::string> _orderids`：订单ID集合，用于标记是否已处理过该订单
  - `wt_hashmap<std::string, double> _undone_qty`：未完成订单数量映射表，合约代码 ——> 未完成数量（正数表示买入未完成，负数表示卖出未完成）

- **成交与自成交检测**
  - `wt_hashmap<std::string, std::string> _trade_refs`：成交单与订单的匹配关系，key为成交单号，value为订单号（用于自成交检测）
  - `wt_hashset<std::string> _self_matches`：自成交合约集合，记录发生过自成交的合约代码
    - **自成交**：同一账户发出的买单和卖单在交易所撮合成交
  - `bool _ignore_sefmatch`：忽略自成交限制标志，true表示允许自成交（自成交发生以后可以恢复交易）

- **交易统计**
  - `TradeStatMap* _stat_map`：交易统计数据映射表，用于记录各合约的交易统计信息
    - typedef WTSHashMap\<std::string\> TradeStatMap：合约ID ——> 交易统计信息WTSTradeStateInfo*

- **风控管理**
  - `bool _risk_mon_enabled`：风控监控是否启用标志
  - `RiskParamsMap _risk_params_map`：风控参数映射表，key为品种代码，value为风控参数
    - typedef wt_hashmap\<std::string, `RiskParams`\> RiskParamsMap
      ```cpp
      /* 风控参数结构体，定义交易风控策略的参数 */
      typedef struct _RiskParams
      {
        uint32_t _order_times_boundary; // 下单频率边界：在统计时间窗口内允许的最大下单次数
        uint32_t _order_stat_timespan;  // 下单统计时间窗口：统计下单频率的时间跨度（秒）
        uint32_t _order_total_limits; // 下单总限额：当日允许的最大下单总次数
        uint32_t _cancel_times_boundary;  // 撤单频率边界：在统计时间窗口内允许的最大撤单次数
        uint32_t _cancel_stat_timespan; // 撤单统计时间窗口：统计撤单频率的时间跨度（秒）
        uint32_t _cancel_total_limits;  // 撤单总限额：当日允许的最大撤单总次数
      } RiskParams;
      ```
  - `CodeTimeCacheMap _order_time_cache`：下单时间缓存，记录每个合约的下单时间戳，用于频率控制
  - `CodeTimeCacheMap _cancel_time_cache`：撤单时间缓存，记录每个合约的撤单时间戳，用于频率控制
    - typedef wt_hashmap\<std::string, `TimeCacheList`\> CodeTimeCacheMap
    - typedef std::vector\<uint64_t\> TimeCacheList（时间戳列表类型）
  - `wt_hashset<std::string> _exclude_codes`：被风控排除的合约集合，这些合约将被禁止交易

- **数据持久化**
  - `bool _save_data`：是否保存交易日志标志
  - `BoostFilePtr _trades_log`：成交数据日志文件指针（CSV格式，包含成交信息）
  - `BoostFilePtr _orders_log`：订单数据日志文件指针（CSV格式，包含订单信息）
  - `std::string _rt_data_file`：实时数据文件路径（JSON格式，包含持仓和资金信息，路径为traders/{交易通道ID}/rtdata.json）

#### 生命周期管理

##### 初始化交易适配器（从配置文件加载）init

##### 初始化交易适配器（使用外部API） initExt

##### 启动交易适配器 run
```cpp
bool TraderAdapter::run()
{
	if (_trader_api == NULL)
		return false;
	if (_stat_map == NULL)
        // 创建统计数据映射表，合约ID ——> 交易统计信息WTSTradeStateInfo*
		_stat_map = TradeStatMap::create();

	_trader_api->registerSpi(this); // 注册回调接口，接收交易服务器的回调通知
	_trader_api->connect(); // 连接交易服务器
	_state = AS_LOGINING; // 设置状态为正在登录
	return true;
}
```

##### 添加交易通知接收器 addSink
```cpp
/* @param sink 交易通知接收器指针，用于接收交易状态变化通知 */
void addSink(ITrdNotifySink* sink)
{
    _sinks.insert(sink);
}
```

#### 查询接口

##### 查询资金信息 queryFund
```cpp
void TraderAdapter::queryFund()
{
	if (_state != AS_ALLREADY) // 如果交易通道未就绪，直接返回
		return;
	_trader_api->queryAccount(); // 查询资金账户								
}
```

##### 获取持仓数量 getPosition
```cpp
/**
 * @brief 获取持仓数量
 * @param stdCode 标准合约代码
 * @param bValidOnly 是否只返回可用持仓（true=可用持仓，false=总持仓）
 * @param flag 持仓标志（1=多头，2=空头）
 * @return 持仓数量（正数表示多头，负数表示空头）
 */
double TraderAdapter::getPosition(const char* stdCode, bool bValidOnly, int32_t flag /* = 3 */)
{
	auto it = _positions.find(stdCode);
	if (it == _positions.end())
		return 0;

	double ret = 0;
	const PosItem& pItem = it->second;
	if(flag & 1)    // 查询多头持仓
	{
		if(bValidOnly) // 如果只要可用持仓
			ret += (pItem.l_newavail + pItem.l_preavail);	// 累加多头可用持仓（今仓可用+昨仓可用）
		else // 如果要总持仓
			ret += (pItem.l_newvol + pItem.l_prevol);		// 累加多头总持仓（今仓+昨仓）
	}

	if (flag & 2) // 查询空头持仓
	{
		if (bValidOnly) // 如果只要可用持仓
			ret -= (pItem.s_newavail + pItem.s_preavail); // 减去空头可用持仓（今仓可用+昨仓可用）
		else // 如果要总持仓
			ret -= pItem.s_newvol + pItem.s_prevol; // 减去空头总持仓（今仓+昨仓）
	}
	return ret;
}
```

##### 获取订单列表 getOrders
从 `_orders: OrderMap*` 获取 stdCode 对应的订单信息返回
```cpp
/**
 * @brief 获取订单列表
 * @param stdCode 标准合约代码，空字符串表示获取所有订单
 * @return 订单映射表指针，失败返回NULL
 */
OrderMap* TraderAdapter::getOrders(const char* stdCode)
```

##### 获取未完成订单数量 getUndoneQty
```cpp
double getUndoneQty(const char* stdCode)
{
    auto it = _undone_qty.find(stdCode);
    if (it != _undone_qty.end())
        return it->second;

    return 0;
}
```

##### 枚举所有持仓 enumPosition
```cpp
/**
 * @brief 枚举所有持仓
 * @param cb 回调函数，对每个有持仓的合约调用该回调
 */
void TraderAdapter::enumPosition(FuncEnumChnlPosCallBack cb)
{
	for(auto& v : _positions) // 遍历所有持仓
	{
		const char* stdCode = v.first.c_str();
		const PosItem& pItem = v.second;
		if(decimal::gt(pItem.l_prevol + pItem.l_newvol, 0)) // 如果有多头持仓
			cb(stdCode, true, pItem.l_prevol, pItem.l_preavail, pItem.l_newvol, pItem.l_newavail); // 回调多头持仓信息
		if (decimal::gt(pItem.s_prevol + pItem.s_newvol, 0)) // 如果有空头持仓
			cb(stdCode, false, pItem.s_prevol, pItem.s_preavail, pItem.s_newvol, pItem.s_newavail); // 回调空头持仓信息
	}
}
```

#### 风控检查接口

##### 检查合约是否允许交易 isTradeEnabled
```cpp
/**
 * @brief 检查合约是否允许交易
 * @param stdCode 标准合约代码
 * @return true表示允许交易，false表示被风控禁止
 */
bool TraderAdapter::isTradeEnabled(const char* stdCode) const
{
	if (!_risk_mon_enabled) // 如果风控监控未启用，则允许交易
		return true;
    // 如果该合约在排除列表中，则禁止交易
	if (_exclude_codes.find(stdCode) != _exclude_codes.end())	
		return false;
	return true;
}
```

##### 检查撤单限制 checkCancelLimits
检查合约 stdCode 当前是否被允许撤单。过程：
- 如果 !`_risk_mon_enabled` 即风险监控未启用，返回 true
- 如果排除列表 `_exclude_code` 中包含 stdCode，返回 false
- 从 `_risk_params_map` 获取该合约的风控参数 riskPara: RiskParams*
  - 如果不存在：返回 true
  - 否则：
    - 从 `_stat_map` 获取该合约的交易统计信息 statInfo: WTSTradeStateInfo*
    - 如果 stateInfo 的撤单总数不小于 riskPara 的总撤单次数限制_cancel_total_limits
      - `_exclude_code` 加入该合约，返回 false
    - 获取撤单时间缓存 `_cancel_time_cache` 中该合约 *\[最后撤单时间-riskPara._cancel_stat_timespan, 最后撤单时间\]* 中间的撤单次数，如果该次数大于 riskPara 的撤单频率限制_cancel_times_boundary
      - `_exclude_code` 加入该合约，返回 false
    - 清除 `_cancel_time_cache` 中该合约在 *最后撤单时间-riskPara._cancel_stat_timespan* 之前的缓存
- 返回 true
```cpp
/**
 * @brief 检查撤单限制
 * @param stdCode 标准合约代码
 * @return true表示允许撤单，false表示超过限制
 */
bool TraderAdapter::checkCancelLimits(const char* stdCode)
```

##### 检查下单限制 checkOrderLimits
检查合约 stdCode 当前是否被允许下单。过程：
- 如果 !`_risk_mon_enabled` 即风险监控未启用，返回 true
- 如果排除列表 `_exclude_code` 中包含 stdCode，返回 false
- 从 `_risk_params_map` 获取该合约的风控参数 riskPara: RiskParams*
  - 如果不存在：返回 true
  - 否则：
    - 从 `_stat_map` 获取该合约的交易统计信息 statInfo: WTSTradeStateInfo*
    - 如果 stateInfo 的下单总数不小于 riskPara 的总下单次数限制_order_total_limits
      - `_exclude_code` 加入该合约，返回 false
    - 获取下单时间缓存 `_order_time_cache` 中该合约 *\[最后下单时间-riskPara._order_stat_timespan, 最后下单时间\]* 中间的下单次数，如果该次数大于 riskPara 的下单频率限制_order_times_boundary
      - `_exclude_code` 加入该合约，返回 false
    - 清除 `_order_time_cache` 中该合约在 *最后下单时间-riskPara._order_stat_timespan* 之前的缓存
- 返回 true
```cpp
/**
 * @brief 检查下单限制
 * @param stdCode 标准合约代码
 * @return true表示允许下单，false表示超过限制
 */
bool TraderAdapter::checkOrderLimits(const char* stdCode)
```

##### 检查是否自成交 checkSelfMatch
将成交信息 tInfo 的 *成交单号——>关联订单号* 存到 `_trade_refs`。如果 `_trade_refs` 已经存在对应成交单号，则检查是否自交
- 存储的关联订单号和 tInfo 的关联订单号不同，即是产生了自交
```cpp
/**
 * @brief 检查自成交
 * @param stdCode 标准合约代码
 * @param tInfo 成交信息对象
 * @return true表示检测到自成交，false表示未检测到自成交
 */
bool TraderAdapter::checkSelfMatch(const char* stdCode, WTSTradeInfo* tInfo)
{
	if (tInfo == NULL)
		return false;

	const char* tid = tInfo->getTradeID();  // 成交单号
	const char* refid = tInfo->getRefOrder();   // 关联订单号

	auto it = _trade_refs.find(tid);
	if (it != _trade_refs.end())
	{
		const std::string& oid = it->second;    // 获取已保存的关联订单号
		if (oid.compare(refid) != 0)    // 如果已经保存的关联订单号和 tInfo 的关联订单号不同，说明自交
		{
            // 同一个成交单号对应不同的订单号 = 自成交！
            // 说明：同一个成交单T001，第一次推送时关联订单是O001（买单）
            //       第二次推送时关联订单是O002（卖单）
            //       说明O001和O002都是本账户的订单，发生了自成交
			WTSLogger::log_dyn("trader", _id.c_str(), LL_FATAL, 
				"[{0}] Self matching detected on {1}!!! Instructions on {1} will be forbidden!!!", _id.c_str(), stdCode);
			_self_matches.insert(stdCode);  // 将该合约加入自成交列表
			return true;    // 返回true表示检测到自成交
		}
		else
		{
			// 关联订单一样，说明是重复推送，不用管了
		}
	}
	else    // 如果成交单号不存在
	{
		_trade_refs[tid] = refid;   // 保存成交单号和关联订单号的映射
	}

	return false;													// 返回false表示未检测到自成交
}
```

##### 检查合约是否在自成交名单中 isSelfMatched
```cpp
/**
 * @brief 检查合约是否在自成交名单中
 * @param stdCode 标准合约代码
 * @return true表示该合约发生过自成交，被禁止交易
 */
inline	bool isSelfMatched(const char* stdCode)
{
    // 如果忽略自成交，则直接返回false
    if (_ignore_sefmatch)
        return false;

    auto it = _self_matches.find(stdCode);
    return it != _self_matches.end();
}
```

#### 辅助方法

##### 生成本地订单ID makeLocalOrderID
- 首次调用时：基于当前时间距离年初的秒数，乘以50作为初始值
- 后续通过原子递增生成唯一ID
```cpp
uint32_t makeLocalOrderID()
{
	static std::atomic<uint32_t> _auto_order_id{ 0 };
	if (_auto_order_id == 0)
	{
        // 计算当前年份的第一天（YYYY0101格式）
		uint32_t curYear = TimeUtils::getCurDate() / 10000 * 10000 + 101;
        // 计算距离年初的秒数，乘以50作为初始ID
		_auto_order_id = (uint32_t)((TimeUtils::getLocalTimeNow() - TimeUtils::makeTime(curYear, 0)) / 1000 * 50);
	}
    // 原子递增并返回新ID
	return _auto_order_id.fetch_add(1);			
}
```

#### 交易操作接口

##### 执行委托下单 doEntrust
具体过程：
- 交易接口 `_trader_api` 调用 makeEntrustID 生成委托单号给 entrust->m_strEntrustID
- 将 `{_order_pattern}.{localid}` 设置给 entrust->m_strUserTag
  - 本地订单ID localid 由函数 makeLocalOrderID 生成
- 交易接口调用 `_trader_api->orderInsert(entrust)` 进行下单
  - 下单失败：返回 UINT_MAX
  - 下单成功：
    - 在下单时间缓存 `_order_time_cache` 对应 entrust 的合约的向量的尾部，添加当前时间
    - 返回 localid
```cpp
/**
 * @brief 执行委托下单
 * @param entrust 委托单对象，包含合约、价格、数量等信息
 * @return 本地订单ID，失败返回UINT_MAX
 */
uint32_t TraderAdapter::doEntrust(WTSEntrust* entrust)
```

##### 执行撤单操作 doCancel
具体过程：
- 如果订单 ordInfo 为 *全部成交/已撤销* 状态
  - 返回 false
- 根据 *期权类型*/*期货*/*其它类型*，生成合约标准代码 stdCode
- 使用 checkCancelLimits 检查合约 stdCode 当前是否被允许撤单
  - 如果不被允许，返回 false
- 创建撤单动作对象 action: WTSEntrustAction*
  - 将 ordInfo 的委托单号 m_strEntrustID 和订单号 m_strOrderID 设置给它
- 交易接口 `_trader_api` 调用 orderAction(action) 进行撤单
  - 撤单失败返回 false，否则返回 true
```cpp
/**
 * @brief 执行撤单操作
 * @param ordInfo 订单信息对象
 * @return true表示撤单请求已发送，false表示失败
 */
bool TraderAdapter::doCancel(WTSOrderInfo* ordInfo)
```

##### 开多仓 openLong
- 使用 stdCode, qty, price 创建一个委托单 entrust: WTSEntrust*
  - 根据 price 是否为 0 将 m_priceType 设置为 *市价单*/*限价单*
  - 根据 flag 设置 m_orderFlag
  - 设置 m_direction 为多头，m_offsetType 为开仓
- 调用 `doEntrust(entrust)` 执行委托下单
```cpp
/**
 * @brief 开多仓
 * @param stdCode 标准合约代码
 * @param price 委托价格，0表示市价
 * @param qty 委托数量
 * @param flag 订单标志（用于扩展订单属性）
 * @param cInfo 合约信息，可为NULL（会自动获取）
 * @return 本地订单ID，失败返回UINT_MAX
 */
uint32_t TraderAdapter::openLong(const char* stdCode, double price, double qty, int flag, WTSContractInfo* cInfo /* = NULL */)
```

##### 开空仓 openShort
- 使用 stdCode, qty, price 创建一个委托单 entrust: WTSEntrust*
  - 根据 price 是否为 0 将 m_priceType 设置为 *市价单*/*限价单*
  - 根据 flag 设置 m_orderFlag
  - 设置 m_direction 为空头，m_offsetType 为开仓
- 调用 doEntrust(entrust) 执行委托下单
```cpp
/**
 * @brief 开空仓
 * @param stdCode 标准合约代码
 * @param price 委托价格，0表示市价
 * @param qty 委托数量
 * @param flag 订单标志（用于扩展订单属性）
 * @param cInfo 合约信息，可为NULL（会自动获取）
 * @return 本地订单ID，失败返回UINT_MAX
 */
uint32_t TraderAdapter::openShort(const char* stdCode, double price, double qty, int flag, WTSContractInfo* cInfo /* = NULL */)
```

##### 平多仓 closeLong
- 使用 stdCode, qty, price 创建一个委托单 entrust: WTSEntrust*
  - 根据 price 是否为 0 将 m_priceType 设置为 *市价单*/*限价单*
  - 根据 flag 设置 m_orderFlag
  - 设置 m_direction 为多头
  - 根据 isToday 设置 m_offsetType 为 *平今*/*平昨*
- 调用 doEntrust(entrust) 执行委托下单
```cpp
/**
 * @brief 平多仓
 * @param stdCode 标准合约代码
 * @param price 委托价格，0表示市价
 * @param qty 委托数量
 * @param isToday 是否平今仓（true=平今仓，false=平昨仓）
 * @param flag 订单标志（用于扩展订单属性）
 * @param cInfo 合约信息，可为NULL（会自动获取）
 * @return 本地订单ID，失败返回UINT_MAX
 */
uint32_t TraderAdapter::closeLong(const char* stdCode, double price, double qty, bool isToday, int flag, WTSContractInfo* cInfo /* = NULL */)
```

##### 平空仓 closeShort
- 使用 stdCode, qty, price 创建一个委托单 entrust: WTSEntrust*
  - 根据 price 是否为 0 将 m_priceType 设置为 *市价单*/*限价单*
  - 根据 flag 设置 m_orderFlag
  - 设置 m_direction 为空头
  - 根据 isToday 设置 m_offsetType 为 *平今*/*平昨*
- 调用 doEntrust(entrust) 执行委托下单
```cpp
/**
 * @brief 平空仓
 * @param stdCode 标准合约代码
 * @param price 委托价格，0表示市价
 * @param qty 委托数量
 * @param isToday 是否平今仓（true=平今仓，false=平昨仓）
 * @param flag 订单标志（用于扩展订单属性）
 * @param cInfo 合约信息，可为NULL（会自动获取）
 * @return 本地订单ID，失败返回UINT_MAX
 */
uint32_t TraderAdapter::closeShort(const char* stdCode, double price, double qty, bool isToday, int flag, WTSContractInfo* cInfo /* = NULL */)
```

##### 撤销指定订单 cancel
- 从订单映射表 `_orders` 中获取对应 localid 的订单
- 调用 doCancel 进行撤单
- 在 `_cancel_time_cache` 对应该合约的向量中记录当前时间（最新撤单时间）
```cpp
/**
 * @brief 根据本地订单ID撤单
 * @param localid 本地订单ID
 * @return true表示撤单请求已发送，false表示失败
 */
bool TraderAdapter::cancel(uint32_t localid)
```

##### 撤销指定合约的订单 cancel
- 根据标准合约代码 stdCode 提取合约信息 cInfo
  - isAll: stdCode 为空则表示撤销所有合约的订单
- 遍历订单映射表 `_orders`
  - 如果该订单还没结束（非 *全部成交*/*已撤销*）
    - 判断是买单（多头开仓或空头平仓）还是卖单（空头开仓或多头平仓），如果与 isBuy 一致
      - 如果 isAll 或该订单对应的合约与 cInfo 一致
        - 调用 `doCancel` 进行撤单，如果撤单成功
          - 累加该订单的撤单数量 m_dVolLeft 到 actQty，并在 ret 尾部记录订单ID
      - 如果已撤单数量 actQty 大于 qty：跳出循环
- 返回 ret
```cpp
/**
 * @brief 批量撤单（按合约和方向）
 * @param stdCode 标准合约代码，空字符串表示撤所有合约的订单
 * @param isBuy 是否买入方向（true=撤买单，false=撤卖单）
 * @param qty 撤单数量，0表示撤所有符合条件的订单
 * @return 已撤单的本地订单ID列表
 */
OrderIDs TraderAdapter::cancel(const char* stdCode, bool isBuy, double qty /* = 0 */)
```

##### 买入操作（智能开平） buy
```cpp
/**
 * @brief 买入操作（智能开平）
 * @param stdCode 标准合约代码
 * @param price 委托价格，0表示市价
 * @param qty 委托数量
 * @param flag 订单标志（用于扩展订单属性）
 * @param bForceClose 是否强制平仓（true=优先平仓，false=优先开仓）
 * @param cInfo 合约信息，可为NULL（会自动获取）
 * @return 订单ID列表
 */
OrderIDs TraderAdapter::buy(const char* stdCode, double price, double qty, int flag, bool bForceClose, WTSContractInfo* cInfo /* = NULL */)
```
注意是 *买入*：即多头开仓，或空头平仓。流程：
- **检查与初始化**
  - 委托数量 qty 为 0：直接返回
  - 如果 stdCode 对应的合约发生过自成交：直接返回
  - 如果当前时间不在 stdCode 对应的交易时段模板的交易时间内：直接返回
  - 将 qty 累加到 `_undone_qty[stdCode]`
  - 根据 *市价单*/*限价单*（price 的值）确定单笔委托单最大交易数量 unitQty
  - 从 `_stat_map` 中获取**对应 stdCode 的交易统计信息 statItem: TradeStatInfo**
  - 从动作策略管理器 `_policy_mgr` 获取**对应 stdCode 的动作规则组 ruleGP**
  - 初始化剩余待处理数量 left = qty
- **遍历动作规则组 ruleGP，对于当前规则 curRule**（直到 left 为 0）：
  - **开仓（且 !bForceClose 即非强平）**
    - 如果 stateItem 保存的当日开多头数 l_openvol 不小于 curRule 规定的当日多头方向手数 _limit_l
      - 跳过该规则
    - 否则：
      - 剩余可开仓数量 maxQty = min(left, curRule._limit_l - stateItem.l_openvol)
    - 如果 stateItem 保存的当日总开仓数（l_openvol + s_openvol）不小于 curRule 规定的当日手数 _limit
      - 跳过该规则
    - 否则：
      - 剩余可开仓数量 maxQty = min(maxQty, curRule._limit - statItem.l_openvol - statItem.s_openvol)
    - 待下单数量 leftQty = maxQty
    - **循环下单（直到 leftQty 为 0 时跳出）**：
      - 使用单笔数量 curQty = min(maxQty, unitQty)，调用 `openLong` 进行实际开多仓操作
      - leftQty -= curQty，本地订单ID添加到 ret 尾部
    - left -= maxQty
  - **平空头今仓**
    - 如果该合约支持平今：
      - maxQty = min(left, 空头今仓 _positions[stdCode].s_newavail)
    - 否则：
      - maxQty = min(left, 空头总仓 _positions[stdCode].s_newavail + _positions[stdCode].s_preavail)
    - 如果非强平 !bForceClose && 空头昨仓_positions[stdCode].s_prevol不为0 && curRule._pure
      - 跳过该规则
    - 待下单数量 leftQty = maxQty
    - **循环下单（直到 leftQty 为 0 时跳出）**：
      - 使用单笔数量 curQty = min(maxQty, unitQty)，调用 `closeShort` 进行实际平今仓操作
      - leftQty -= curQty，本地订单ID添加到 ret 尾部
    - left -= maxQty
  - **平空头昨仓**
    - maxQty = min(left, 空头昨仓 pItem.s_preavail)
    - 如果非强平 !bForceClose && 空头今仓_positions[stdCode].s_newvol不为0 && curRule._pure
      - 跳过该规则
    - 待下单数量 leftQty = maxQty
    - **循环下单（直到 leftQty 为 0 时跳出）**：
      - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeShort` 进行实际平仓操作
      - leftQty -= curQty，本地订单ID添加到 ret 尾部
    - left -= maxQty
  - **平空仓**
    - 如果该合约不支持平今：
      - maxQty = min(left, 空头总仓 _positions[stdCode].s_newavail + _positions[stdCode].s_preavail)
      - 待下单数量 leftQty = maxQty
      - **循环下单（直到 leftQty 为 0 时跳出）**：
        - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeShort` 进行实际平仓操作
        - leftQty -= curQty，本地订单ID添加到 ret 尾部
      - left -= maxQty
    - 否则：
      - maxQty = min(qty, 可用空头昨仓_positions[stdCode].s_preavail)
      - **循环下单（直到 leftQty 为 0 时跳出）**：
        - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeShort` 进行实际平仓操作
        - leftQty -= curQty，本地订单ID添加到 ret 尾部
      - left -= maxQty
      - 如果 left > 0 并且空头今仓 _positions[stdCode].s_newavail > 0
        - maxQty = min(空头今仓 _positions[stdCode].s_newavail, left)
        - **循环下单（直到 leftQty 为 0 时跳出）**：
          - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeShort` 进行实际平仓操作
          - leftQty -= curQty，本地订单ID添加到 ret 尾部
        - left -= maxQty
- 返回 ret

##### 卖出操作（智能开平） sell
```cpp
/**
 * @brief 卖出操作（智能开平）
 * @param stdCode 标准合约代码
 * @param price 委托价格，0表示市价
 * @param qty 委托数量
 * @param flag 订单标志（用于扩展订单属性）
 * @param bForceClose 是否强制平仓（true=优先平仓，false=优先开仓）
 * @param cInfo 合约信息，可为NULL（会自动获取）
 * @return 订单ID列表
 */
OrderIDs TraderAdapter::sell(const char* stdCode, double price, double qty, int flag, bool bForceClose, WTSContractInfo* cInfo /* = NULL */)
```
注意是 *卖出*：即多头平仓，或空头开仓。流程：
- **检查与初始化**
  - 委托数量 qty 为 0：直接返回
  - 如果 stdCode 对应的合约发生过自成交：直接返回
  - 如果当前时间不在 stdCode 对应的交易时段模板的交易时间内：直接返回
  - 将 -qty 累加到 `_undone_qty[stdCode]`
  - 根据 *市价单*/*限价单*（price 的值）确定单笔委托单最大交易数量 unitQty
  - 从 `_stat_map` 中获取**对应 stdCode 的交易统计信息 statItem: TradeStatInfo**
  - 从动作策略管理器 `_policy_mgr` 获取**对应 stdCode 的动作规则组 ruleGP**
  - 初始化剩余待处理数量 left = qty
- **遍历动作规则组 ruleGP，对于当前规则 curRule**（直到 left 为 0）：
  - **开仓（且 !bForceClose 即非强平）**
    - 如果 stateItem 保存的当日开空头数 s_openvol 不小于 curRule 规定的当日空头方向手数 _limit_s
      - 跳过该规则
    - 否则：
      - 剩余可开仓数量 maxQty = min(left, curRule._limit_s - stateItem.s_openvol)
    - 如果 stateItem 保存的当日总开仓数（l_openvol + s_openvol）不小于 curRule 规定的当日手数 _limit
      - 跳过该规则
    - 否则：
      - 剩余可开仓数量 maxQty = min(maxQty, curRule._limit - statItem.l_openvol - statItem.s_openvol)
    - 待下单数量 leftQty = maxQty
    - **循环下单（直到 leftQty 为 0 时跳出）**：
      - 使用单笔数量 curQty = min(maxQty, unitQty)，调用 `openShort` 进行实际`开空仓`操作
      - leftQty -= curQty，本地订单ID添加到 ret 尾部
    - left -= maxQty
  - **平多头今仓**
    - 如果该合约支持平今：
      - maxQty = min(left, 多头今仓 _positions[stdCode].l_newavail)
    - 否则：
      - maxQty = min(left, 多头总仓 _positions[stdCode].l_newavail + _positions[stdCode].l_preavail)
    - 如果非强平 !bForceClose && 多头昨仓_positions[stdCode].l_prevol不为0 && curRule._pure
      - 跳过该规则
    - 待下单数量 leftQty = maxQty
    - **循环下单（直到 leftQty 为 0 时跳出）**：
      - 使用单笔数量 curQty = min(maxQty, unitQty)，调用 `closeLong` 进行实际`平今仓`操作
      - leftQty -= curQty，本地订单ID添加到 ret 尾部
    - left -= maxQty
  - **平多头昨仓**
    - maxQty = min(left, 多头昨仓 pItem.l_preavail)
    - 如果非强平 !bForceClose && 多头今仓_positions[stdCode].l_newvol不为0 && curRule._pure
      - 跳过该规则
    - 待下单数量 leftQty = maxQty
    - **循环下单（直到 leftQty 为 0 时跳出）**：
      - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeLong` 进行实际`平仓`操作
      - leftQty -= curQty，本地订单ID添加到 ret 尾部
    - left -= maxQty
  - **平多头**
    - 如果该合约不支持平今：
      - maxQty = min(left, 多头总仓 _positions[stdCode].l_newavail + _positions[stdCode].l_preavail)
      - 待下单数量 leftQty = maxQty
      - **循环下单（直到 leftQty 为 0 时跳出）**：
        - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeLong` 进行实际`平仓`操作
        - leftQty -= curQty，本地订单ID添加到 ret 尾部
      - left -= maxQty
    - 否则：
      - maxQty = min(qty, 可用多头昨仓_positions[stdCode].l_preavail)
      - **循环下单（直到 leftQty 为 0 时跳出）**：
        - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeLong` 进行实际`平仓`操作
        - leftQty -= curQty，本地订单ID添加到 ret 尾部
      - left -= maxQty
      - 如果 left > 0 并且多头今仓 _positions[stdCode].l_newavail > 0
        - maxQty = min(多头今仓 _positions[stdCode].l_newavail, left)
        - **循环下单（直到 leftQty 为 0 时跳出）**：
          - 使用单笔数量 maxQty = min(maxQty, unitQty)，调用 `closeLong` 进行实际平仓操作
          - leftQty -= curQty，本地订单ID添加到 ret 尾部
        - left -= maxQty
- 返回 ret

#### ITraderSpi接口实现

##### 处理交易事件 handleEvent
```cpp
/**
 * @brief 处理交易事件
 * @param e 交易事件类型
 * @param ec 错误代码
 * 
 * 处理交易通道的连接和断开事件：
 * - WTE_Connect: 连接成功时自动登录，连接失败时记录错误
 * - WTE_Close: 连接断开时通知所有监听器
 */
void TraderAdapter::handleEvent(WTSTraderEvent e, int32_t ec)
{
	if(e == WTE_Connect) // 如果是连接事件
	{
		if(ec == 0) // 如果连接成功
		{
            // 自动登录
			_trader_api->login(_cfg->getCString("user"), _cfg->getCString("pass"), WT_PRODUCT);
		}
		else // 如果连接失败
		{
			WTSLogger::log_dyn("trader", _id.c_str(), LL_ERROR,"[{}] Trading channel connecting failed: {}", _id.c_str(), ec);
		}
	}
	else if(e == WTE_Close) // 如果是断开事件
	{
		WTSLogger::log_dyn("trader", _id.c_str(), LL_ERROR,"[{}] Trading channel disconnected: {}", _id.c_str(), ec);
        // 通知所有监听器，调用通道丢失回调
		for (auto sink : _sinks)									
			sink->on_channel_lost();
	}
}
```

##### 登录结果回调 onLoginResult
```cpp
/**
 * @brief 登录结果回调
 * @param bSucc 是否登录成功
 * @param msg 登录消息（成功或失败原因）
 * @param tradingdate 交易日期
 * 
 * 处理登录结果：
 * - 登录成功：设置状态为已登录，保存交易日期，查询持仓
 * - 登录失败：设置状态为登录失败，记录错误日志，发送通知
 */
void TraderAdapter::onLoginResult(bool bSucc, const char* msg, uint32_t tradingdate)
{
	if(!bSucc) // 如果登录失败
	{
		_state = AS_LOGINFAILED; // 设置状态为登录失败
		WTSLogger::log_dyn("trader", _id.c_str(), LL_ERROR,"[{}] Trader login failed: {}", _id.c_str(), msg);
        // 如果事件通知器存在，发送登录失败通知
		if (_notifier) 
			_notifier->notify(id(), fmt::format("login failed: {}", msg).c_str());
	}
	else // 如果登录成功
	{
		_state = AS_LOGINED; // 设置状态为已登录
		WTSLogger::log_dyn("trader", _id.c_str(), LL_INFO,"[{}] Trader login succeed, trading date: {}", _id.c_str(), tradingdate);
		_trading_day = tradingdate; // 保存交易日期
        // 查询持仓
		_trader_api->queryPositions();
	}
}
```

##### 登出回调 onLogout
```cpp
/**
 * @brief 登出回调 
 * 处理登出事件（当前为空实现）。
 */
void TraderAdapter::onLogout()
{
	
}
```

##### 委托响应回调 onRspEntrust
具体过程：
- 如果委托下单失败（根据 err 判断）：
  - 输出错误日志
  - 如果 `_undone_qty` 中对应的未完成数量为 0
    - 更新 `_undone_qty` 中对应的未完成数量
    - 如果委托单的用户标签 entrust.m_strUserTag 不为空（说明是本系统的委托单）
      - 所有监听器 `_sinks` 调用 on_entrust 回调
      - 通知器 `_notifier` 调用 notify
```cpp
/**
 * @brief 委托响应回调
 * @param entrust 委托单对象
 * @param err 错误信息，成功时为NULL
 * 
 * 处理委托下单的响应：
 * - 如果下单失败：更新未完成订单数量，通知监听器，发送错误通知
 * - 如果下单成功：不做处理（由订单推送回调处理）
 */
void TraderAdapter::onRspEntrust(WTSEntrust* entrust, WTSError *err)
```

##### 资金查询响应回调 onRspAccount
```cpp
/**
 * @brief 资金查询响应回调
 * @param ayAccounts 资金账户信息数组
 * 
 * 处理资金查询响应：
 * 1. 保存资金数据到文件（如果启用了数据保存）
 * 2. 通知所有监听器资金变化
 * 3. 如果所有查询完成，设置状态为就绪，通知监听器通道就绪
 */
void TraderAdapter::onRspAccount(WTSArray* ayAccounts)
{
    // 如果启用了数据保存
	if (_save_data)
	{
		saveData(ayAccounts); // 保存资金数据到文件
	}

	if(ayAccounts)
	{
		// 通知所有监听接口
		for (auto sink : _sinks)
		{
            // 遍历所有资金账户
			for (uint32_t idx = 0; idx < ayAccounts->size(); idx++)
			{
                // 获取资金账户信息
				WTSAccountInfo* fundInfo = (WTSAccountInfo*)ayAccounts->at(idx);	
                // 调用资金回调
				sink->on_account(fundInfo->getCurrency(), fundInfo->getPreBalance(), fundInfo->getBalance(), fundInfo->getBalance() + fundInfo->getDynProfit(), fundInfo->getAvailable(),
					fundInfo->getCloseProfit(), fundInfo->getDynProfit(), fundInfo->getMargin(), fundInfo->getCommission(), fundInfo->getDeposit(), fundInfo->getWithdraw());	
			}
		}
	}
    // 如果成交查询已完成（说明所有查询都完成了）
	if(_state == AS_TRADES_QRYED)				
	{
		_state = AS_ALLREADY; // 设置状态为全部就绪
		WTSLogger::log_dyn("trader", _id.c_str(), LL_INFO, "[{}] Trading channel ready", _id.c_str());
        // 通知所有监听器，调用通道就绪回调
		for (auto& sink : _sinks)
			sink->on_channel_ready();
	}
}
```

##### 持仓查询响应回调 onRspPosition
具体过程：
- 遍历 ayPositions：更新 `_positions` 中对应合约的持仓
- 遍历 `_positions` 中的所有持仓：所有监听器 `_sinks` 调用 on_positions 通知多头/空头持仓
- 如果状态 `_state` 为已登录：
  - `_state` 切换为持仓查询完成
  - 交易接口 `_trader_api` 调用 queryOrders
```cpp
/**
 * @brief 持仓查询响应回调
 * @param ayPositions 持仓信息数组
 * 
 * 处理持仓查询响应：
 * 1. 更新内部持仓数据
 * 2. 打印持仓信息
 * 3. 通知所有监听器持仓变化
 * 4. 如果登录完成，设置状态为持仓查询完成，查询订单
 */
void TraderAdapter::onRspPosition(const WTSArray* ayPositions)
```

##### 订单查询响应回调 onRspOrders
具体过程：
- 清空未完成订单数量映射表 `_undone_qty`
- 遍历 ayOrders 中所有订单 orderInfo: WTSOrderInfo*：
  - 从 `_stat_map` 找到对应合约的交易统计 statItem: TradeStatInfo
  - 使用 orderInfo 更新 statItem
    - 买入/卖出订单次数和数量
    - 错单次数和数量（区分买入和卖出）
    - 撤单次数和数量（区分普通撤单和自动撤单，区分买入和卖出）
  - 如果 orderInfo 已结束（全部成交/已撤销）|| orderInfo.m_strUserTag 与 `_order_pattern` 不一致
    - 跳过
  - `_orders` 添加 *本地订单ID——>orderInfo*
  - 更新 `_undone_qty` 中对应合约的未完成数量
- 如果状态 `_state` 为持仓查询已完成
  - `_state` 切换为订单查询完成
  - 交易接口 `_trader_api` 调用 queryTrades
```cpp
/**
 * @brief 订单查询响应回调
 * @param ayOrders 订单信息数组
 */
void TraderAdapter::onRspOrders(const WTSArray* ayOrders)
```

##### 成交查询响应回调 onRspTrades
具体过程：
- 遍历 ayTrades 中所有成交 tInfo: WTSTradeInfo*：
  - 从 `_stat_map` 找到对应合约的交易统计 statItem: TradeStatInfo
  - 使用 tInfo 更新 statItem
  - 调用 checkSelfMatch 检查该合约是否发生自成交
- 如果状态 `_state` 为订单查询已完成
  - `_state` 切换为成交查询完成
  - 交易接口 `_trader_api` 调用 queryAccount
```cpp
/**
 * @brief 成交查询响应回调
 * @param ayTrades 成交信息数组
 */
void TraderAdapter::onRspTrades(const WTSArray* ayTrades)
```

##### 持仓项 `PosItem` 中 `vol` 和 `avail` 后缀字段的区别

| 字段后缀 | 含义 | 英文全称 |
|---------|------|---------|
| **`vol`** | 总持仓数量 | Volume |
| **`avail`** | 可用持仓数量 | Available |

- `vol`：总持仓数量
  - 仅在成交时变化，反映账户实际持仓
- `avail`：可用持仓数量
  - 成交时变化（***T+非 0* 时，刚成交时不会变化，因为相当于被冻结了**）
  - 平仓订单推送时变化，撤销时恢复（防止重复平仓，控制可平仓数量）
- 可用持仓 = 总持仓 - 已冻结持仓
- 已冻结持仓 = 已提交但未成交的平仓订单数量

##### 订单推送回调（实时推送） onPushOrder
```cpp
/**
 * @brief 订单推送回调
 * @param orderInfo 订单信息对象
 */
void TraderAdapter::onPushOrder(WTSOrderInfo* orderInfo)
```
具体过程：
- **如果订单 orderInfo 为已撤销状态**
  - 更新 `_stat_map` 中对应合约的 *错单*/*撤单* 挂单数据
- **检查该订单是第一次推送（`_orderids` 中是否包含它）**
  - **是**
    - 将订单号插入到 `_orderids`
    - **更新 `_stat_map` 中对应的 statItem 的正常挂单数据**
      - 买单：statItem.b_orders++、statItem.b_ordqty += orderInfo.m_dVolume
      - 卖单：statItem.s_orders++、statItem.s_ordqty += orderInfo.m_dVolume
    - **如果 orderInfo 是平仓单，更新 `_positions` 中对应的 pItem 的可用（avail）项**
      - 平多今仓：多今仓 -= min(多今仓，orderInfo的订单数量)
      - 平多昨仓：
        - 多昨仓 -= min(多昨仓, qty)，多昨仓 = max(多昨仓, 0)
        - 如果 qty > 多昨仓（需要多今仓来平）
          - 多今仓 -= min(多今仓，qty - 多昨仓)，多今仓 = max(多今仓, 0)
      - 平空今仓：空今仓 -= min(空今仓，orderInfo的订单数量)
      - 平空昨仓：
        - 空昨仓 -= min(空昨仓, qty)，空昨仓 = max(空昨仓, 0)
        - 如果 qty > 空昨仓（需要空今仓来平）
          - 空今仓 -= min(空今仓，qty - 空昨仓)，空今仓 = max(空今仓, 0)
  - **否则如果：orderInfo 撤销了 && 是平仓单** 
    - 平多今仓：多今仓 += orderInfo的订单数量
    - 平多昨仓：
      - 多昨仓 += qty
      - 如果 多昨仓 > 实际多昨仓
        - 多今仓 += 多昨仓 - 实际多昨仓
        - 多昨仓 = 实际多昨仓
    - 平空今仓：空今仓 += orderInfo的订单数量
    - 平空昨仓：
      - 空昨仓 += qty
      - 如果 空昨仓 > 实际空昨仓
        - 空今仓 += 空昨仓 - 实际空昨仓
        - 空昨仓 = 实际空昨仓
- **orderInfo.m_strUserTag 与 `_order_pattern` 一致（本系统发出的订单）**
  - 调用 updateUndone 更新 `_undone_qty` 中对应合约的未完成数量
  - 如果 orderInfo 已结束 && `_orders` 存在
    - 从 `_orders` 中删除该订单
  - 否则：在 `_orders` 中插入该订单
  - 所有监听器 `_sinks` 调用 on_order
- **如果 `_save_data` 并且 orderInfo 已经结束了**
  - 调用 logOrder 记录订单日志
- **事件通知器 `_notifier` 调用 notify 发送订单通知**

##### 成交推送回调（实时推送） onPushTrade
```cpp
/**
 * @brief 成交推送回调
 * @param tradeRecord 成交信息对象
 */
void TraderAdapter::onPushTrade(WTSTradeInfo* tradeRecord)
```
具体过程：
- 记 tradeRecord 的成交数量为 vol
- **更新交易统计 `_stat_map` 和持仓 `_positions` 中对应项 statItem 和 pItem**：
  - 开多头
    - pItem.多今仓 += vol
    - 如果不是T+1（当日可卖）：pItem.可用多头今仓 += vol
    - statItem.当日开多 += vol
  - 平今多头
    - pItem.多今仓 -= vol
    - statItem.当日平多 += vol
  - 平昨多头
    - pItem.多昨仓 -= min(vol, pItem.多昨仓)
    - pItem.多今仓 -= vol - min(left, pItem.多昨仓) （平昨不足时，用今多仓补充）
    - statItem.当日平多 += vol
  - 开空头
    - pItem.空今仓 += vol
    - 如果不是T+1（当日可卖）：pItem.可用空头今仓 += vol
    - statItem.当日开空 += vol
  - 平今空头
    - pItem.空今仓 -= vol
    - statItem.当日平空 += vol
  - 平昨空头
    - pItem.空昨仓 -= min(vol, pItem.空昨仓)
    - pItem.空今仓 -= vol - min(left, pItem.空昨仓) （平昨不足时，用今空仓补充）
    - statItem.当日平空 += vol
- **orderInfo.m_strUserTag 与 `_order_pattern` 一致（本系统发出的订单）**
  - 调用 updateUndone 更新 `_undone_qty` 中对应合约的未完成数量
- **所有监听器 `_sinks` 调用 on_trade 进行成交回调**
- **如果 `_save_data` 调用 logOrder 记录订单日志**
- **调用 checkSelfMatch 检查自成交**
- **事件通知器 `_notifier` 调用 notify 发送成交通知**
- **`_trader_api` 调用 queryAccount 查询资金账户**

##### 交易错误回调 onTraderError
```cpp
/**
 * @brief 交易错误回调
 * @param err 错误信息对象
 * @param pData 附加数据，可为NULL
 */
void TraderAdapter::onTraderError(WTSError* err, void* pData /* = NULL */)
{
	if(err) // 如果错误信息存在
		WTSLogger::log_dyn("trader", _id.c_str(), LL_ERROR,"[{}] Error of trading channel occured: {}", _id.c_str(), err->getMessage()); // 记录错误日志

	if (_notifier) // 如果事件通知器存在
		_notifier->notify(id(), fmt::format("Trading channel error: {}", err->getMessage()).c_str()); // 发送错误通知
}
```

# 数据管理 WtDtMgr.h/cpp
负责管理策略运行所需的各种数据，包括K线数据和Tick数据。
- 管理K线数据缓存，支持多周期K线重采样
- 管理Tick数据缓存，支持实时Tick和后复权Tick的处理
- 处理K线更新事件，统一通知引擎
- 提供数据订阅管理功能
```cpp
class WtDtMgr : public IDataReaderSink, public IDataManager
```
参考 [Includes/note.ipynb/数据管理接口层](../Includes/note.ipynb/数据管理接口层)

## 成员
- **核心管理器指针**
  - `IDataReader* _reader`：数据读取器指针，用于读取历史数据和实时数据
  - `IHisDataLoader* _loader`：历史数据加载器指针，用于加载历史数据
  - `WtEngine* _engine`：引擎指针，用于获取会话信息和触发事件
- **配置与状态标志**
  - `bool _align_by_section`：强制小节对齐标志，true表示K线重采样时按小节对齐，false表示按时间对齐
  - `bool _force_cache`：强制缓存K线标志，true表示所有K线都缓存（包括1倍周期），false表示只有重采样K线缓存
- **数据订阅管理**
  - `wt_hashset<std::string> _subed_basic_bars`：已订阅的**基础周期**K线集合
    - 键为"合约代码-周期"格式（如"SHFE.ag.1912-KP_Minute1"）
- **K线数据缓存**
  - `DataCacheMap* _bars_cache`：K线缓存映射表
    - typedef `WTSHashMap<std::string>` DataCacheMap：数据缓存映射表类型定义
      - 键格式：`"合约代码-周期-倍数"`（如"SHFE.ag.1912-KP_Minute1-5"表示5分钟K线）
      - 值类型：`WTSKlineData*`，存储重采样后的K线数据
- **Tick数据缓存**
  - `DataCacheMap* _rt_tick_map`：实时Tick缓存映射表
    - typedef `WTSHashMap<std::string>` DataCacheMap：数据缓存映射表类型定义
      - 键格式：标准合约代码（如"SHFE.ag.1912"）
      - 值类型：`WTSTickData*`，存储最新的实时Tick数据
  - `DataCacheMap* _ticks_adjusted`：**后复权**Tick缓存映射表
    - typedef `WTSHashMap<std::string>` DataCacheMap：数据缓存映射表类型定义
      - 键格式：合约代码（不含+后缀，如"SHFE.ag.1912"）
      - 值类型：`WTSHisTickData*`，**仅缓存后复权数据（前复权和不复权不需要缓存）**
- **K线通知队列**
  - `std::vector<NotifyItem> _bar_notifies`：K线通知队列，存储待通知的K线更新项
    - `NotifyItem`：K线通知项结构体类型定义，包含：
      - `char _code[MAX_INSTRUMENT_LENGTH]`：合约代码字符串（最大长度限制）
      - `char _period[2]`：周期字符（'m'表示分钟，'d'表示日），包含结束符
      - `uint32_t _times`：K线倍数（如1分钟K线的times=1，5分钟K线的times=5）
      - `WTSBarStruct* _newBar`：新的K线数据指针
    - 用途：延迟通知机制，收集所有K线更新后统一触发引擎的on_bar事件，避免重复通知

## 方法

### 初始化与配置管理

#### 初始化数据管理器 init

#### 初始化数据存储模块 initStore

### IDataManager接口实现—数据访问接口

#### 获取Tick数据切片 get_tick_slice
- 如果 `_ticks_adjusted` 中不存在对应的合约
  - 用 `_reader` 读取原始Tick数据
  - 用 `_engine` 读取复权因子
  - 根据复权因子调整原始Tick数据，插入到 `_ticks_adjusted` 中
- 读取 `_ticks_adjusted` 中 etime 之前的 count 个Tick数据，作为 WTSTickSlice* 来返回
```cpp
/**
 * @brief 获取Tick数据切片
 * @param stdCode 标准合约代码字符串
 * @param count 获取的Tick数量
 * @param etime 结束时间戳，默认为0（使用最新时间）
 * @return WTSTickSlice* 返回Tick数据切片指针，如果数据读取器无效返回NULL
 */
WTSTickSlice* WtDtMgr::get_tick_slice(const char* stdCode, uint32_t count, uint64_t etime /* = 0 */)
```

#### 获取订单队列切片 get_order_queue_slice
```cpp
/**
 * @brief 获取订单队列切片
 * @param stdCode 标准合约代码字符串
 * @param count 获取的数据条数
 * @param etime 结束时间戳，默认为0（使用最新时间）
 * @return WTSOrdQueSlice* 返回订单队列切片指针，如果数据读取器无效返回NULL
 */
WTSOrdQueSlice* WtDtMgr::get_order_queue_slice(const char* stdCode, uint32_t count, uint64_t etime /* = 0 */)
{
	if (_reader == NULL)
		return NULL;
	return _reader->readOrdQueSlice(stdCode, count, etime);
}
```

#### 获取订单明细切片 get_order_detail_slice

```cpp
/**
 * @brief 获取订单明细切片
 * @param stdCode 标准合约代码字符串
 * @param count 获取的数据条数
 * @param etime 结束时间戳，默认为0（使用最新时间）
 * @return WTSOrdDtlSlice* 返回订单明细切片指针，如果数据读取器无效返回NULL
 */
WTSOrdDtlSlice* WtDtMgr::get_order_detail_slice(const char* stdCode, uint32_t count, uint64_t etime /* = 0 */)
{
	if (_reader == NULL)
		return NULL;
	return _reader->readOrdDtlSlice(stdCode, count, etime);
}
```

#### 获取逐笔成交切片 get_transaction_slice
```cpp
/**
 * @brief 获取逐笔成交切片
 * @param stdCode 标准合约代码字符串
 * @param count 获取的数据条数
 * @param etime 结束时间戳，默认为0（使用最新时间）
 * @return WTSTransSlice* 返回逐笔成交切片指针，如果数据读取器无效返回NULL
 */
WTSTransSlice* WtDtMgr::get_transaction_slice(const char* stdCode, uint32_t count, uint64_t etime /* = 0 */)
{
	if (_reader == NULL)
		return NULL;
	return _reader->readTransSlice(stdCode, count, etime);
}
```

#### 获取K线数据切片 get_kline_slice
- 如果倍数 times 为 1 并且非强制缓存 !_force_cache
  - 将 *{stdCode}.{period}* 存入 `_subed_basic_bars`
  - `_reader` 调用 readKlineSlice 并返回
- 从K线缓存 `_bars_cache` 中读取对应 *{stdCode}.{period}.{times}* 的K线切片 kData
- 如果 kData 为空或者其数量小于请求数 count
  - `_reader` 调用 readKlineSlice 读取基础（1倍周期）K线数据
  - 根据 times 是否为 1 决定是否调用 `g_dataFact.extractKlineData` 来重采样
- 返回 kData
```cpp
/**
 * @brief 获取K线数据切片
 * @param stdCode 标准合约代码字符串
 * @param period K线周期
 * @param times K线倍数
 * @param count 获取的K线数量
 * @param etime 结束时间戳，默认为0（使用最新时间）
 * @return WTSKlineSlice* 返回K线数据切片指针，如果数据读取器无效返回NULL
 */
WTSKlineSlice* WtDtMgr::get_kline_slice(const char* stdCode, WTSKlinePeriod period, uint32_t times, uint32_t count, uint64_t etime /* = 0 */)
```

#### 从实时缓存中获取最新Tick数据 grab_last_tick
```cpp
/**
 * @brief 获取最后一个Tick数据
 * @param stdCode 标准合约代码字符串
 * @return WTSTickData* 返回最新的Tick数据指针，如果缓存无效返回NULL
 */
WTSTickData* WtDtMgr::grab_last_tick(const char* code)
{
	if (_rt_tick_map == NULL)
		return NULL;

	WTSTickData* curTick = (WTSTickData*)_rt_tick_map->get(code);
	if (curTick == NULL)
		return NULL;

	curTick->retain();
	return curTick;
}
```

#### 获取复权因子 get_adjusting_factor
```cpp
/**
 * @brief 获取复权因子
 * @param stdCode 标准合约代码字符串
 * @param uDate 日期（格式：YYYYMMDD）
 * @return double 返回复权因子，如果数据读取器无效返回1.0
 */
double WtDtMgr::get_adjusting_factor(const char* stdCode, uint32_t uDate)
{
	if (_reader)
        // 从数据读取器获取指定日期的复权因子
		return _reader->getAdjFactorByDate(stdCode, uDate);
	return 1.0;
}
```

#### 获取复权标志 get_adjusting_flag
```cpp
/**
 * @brief 获取复权标志
 * @return uint32_t 返回复权标志（0=不复权，1=前复权，2=后复权）
 */
uint32_t WtDtMgr::get_adjusting_flag()
{
	static uint32_t flag = UINT_MAX;  // 静态变量，用于缓存复权标志（初始值为UINT_MAX表示未初始化）
	if(flag == UINT_MAX)  // 如果标志未初始化
	{
		if (_reader)
			flag = _reader->getAdjustingFlag();
		else 
			flag = 0;
	}
	return flag;
}
```

### IDataReaderSink接口实现—数据读取器回调

#### K线更新回调 on_bar
- 如果 newBar 更新的是基础周期
  - 如果 `_bars_cache` 中能找到对应的：
    - 将 newBar 添加到其最后
    - 构建K线更新项加入通知队列 `_bar_notifies`
- 否则
  - 如果 `_bars_cache` 中能找到对应的K线缓存
    - 调用 `g_dataFact.updateKlineData` 来用 newBar 更新
    - 如果该K线闭合：构建K线更新项加入通知队列 `_bar_notifies`
```cpp
/**
 * @brief K线更新回调
 * @param code 合约代码字符串
 * @param period K线周期
 * @param newBar 新的K线数据指针
 */
void WtDtMgr::on_bar(const char* code, WTSKlinePeriod period, WTSBarStruct* newBar)
```

#### 所有K线更新完成回调 on_all_bar_updated
```cpp
/**
 * @brief 所有K线更新完成回调
 * @param updateTime 更新时间戳
 * 
 * 当数据读取器完成所有K线更新时被调用。
 * 统一处理通知队列中的所有K线更新事件，触发引擎的on_bar事件。
 */
void WtDtMgr::on_all_bar_updated(uint32_t updateTime)
{
	if (_bar_notifies.empty())
		return;
	WTSLogger::debug("All bars updated, on_bar will be triggered");

	for (const NotifyItem& item : _bar_notifies) // 遍历通知队列
	{
        // 触发引擎的on_bar事件（传递合约代码、周期、倍数和K线数据）
		_engine->on_bar(item._code, item._period, item._times, item._newBar);  
	}
	_bar_notifies.clear();
}
```

#### 获取当前日期 get_date
```cpp
/**
 * @brief 获取当前日期
 * @return uint32_t 返回当前日期（格式：YYYYMMDD）
 */
uint32_t WtDtMgr::get_date() 
{ 
	return _engine->get_date();  // 从引擎获取当前日期
}
```

#### 获取当前分钟时间 get_min_time
```cpp
/**
 * @brief 获取当前分钟时间
 * @return uint32_t 返回当前分钟时间（格式：HHMM）
 */
uint32_t WtDtMgr::get_min_time()
{ 
	return _engine->get_min_time();  // 从引擎获取当前分钟时间
}
```

#### 获取当前秒数 get_secs
```cpp
/**
 * @brief 获取当前秒数
 * @return uint32_t 返回当前秒数（包含毫秒，格式：SSmmm）
 */
uint32_t WtDtMgr::get_secs() 
{ 
	return _engine->get_secs();  // 从引擎获取当前秒数
}
```

#### 数据读取器日志回调 reader_log
```cpp
/**
 * @brief 数据读取器日志回调
 * @param ll 日志级别
 * @param message 日志消息字符串
 */
void WtDtMgr::reader_log(WTSLogLevel ll, const char* message)
{
	WTSLogger::log_raw(ll, message);  // 直接记录原始日志（使用指定的日志级别）
}
```

### 数据推送与处理

#### 处理推送的行情数据 handle_push_quote
将 newTick 添加到实时Tick缓存 `_rt_tick_map` 以及历史后复权Tick缓存 `_ticks_adjusted`
```cpp
/**
 * @brief 处理推送的行情数据
 * @param stdCode 标准合约代码字符串
 * @param newTick 新的Tick数据指针
 */
void WtDtMgr::handle_push_quote(const char* stdCode, WTSTickData* newTick)
{
	if (newTick == NULL)
		return;

	if (_rt_tick_map == NULL)
		_rt_tick_map = DataCacheMap::create();
	_rt_tick_map->add(stdCode, newTick, true);

	if(_ticks_adjusted != NULL)
	{
		WTSHisTickData* tData = (WTSHisTickData*)_ticks_adjusted->get(stdCode);
		if (tData == NULL)
			return;

		if (tData->isValidOnly() && newTick->volume() == 0)
		tData->appendTick(newTick->getTickStruct());
	}
}
```

# 引擎层

## WtEngine.h/cpp

```mermaid
graph TB
    %% 接口层
    WtPortContext[WtPortContext<br/>组合上下文接口<br/>提供风控模块所需上下文]
    IParserStub[IParserStub<br/>行情解析存根接口<br/>接收行情数据推送]
    IEngineEvtListener[IEngineEvtListener<br/>引擎事件监听器接口<br/>监听引擎事件]
    
    %% 核心引擎类
    WtEngine[WtEngine<br/>交易引擎基类<br/>核心功能:<br/>- 时间管理<br/>- 数据访问<br/>- 持仓管理<br/>- 资金管理<br/>- 信号处理<br/>- 风控管理<br/>- 任务调度]
    
    %% 继承关系
    WtPortContext -->|实现| WtEngine
    IParserStub -->|实现| WtEngine
    
    %% 风控相关
    WtRiskMonitor[WtRiskMonitor<br/>风控监视器<br/>监控组合风险]
    IRiskMonitorFact[IRiskMonitorFact<br/>风控监视器工厂<br/>创建/删除风控监视器]
    WtRiskMonWrapper[WtRiskMonWrapper<br/>风控监视器包装类<br/>管理生命周期]
    
    WtRiskMonWrapper -->|包含| WtRiskMonitor
    WtRiskMonWrapper -->|使用| IRiskMonitorFact
    WtEngine -->|使用| WtRiskMonWrapper
    
    
    WtEngine -->|注册| IEngineEvtListener
    %% 样式
    classDef interface fill:#e1f5ff,stroke:#01579b,stroke-width:2px
    classDef core fill:#fff3e0,stroke:#e65100,stroke-width:3px
    classDef manager fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef struct fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px
    classDef wrapper fill:#fce4ec,stroke:#880e4f,stroke-width:2px
    
    class WtPortContext,IParserStub,IEngineEvtListener interface
    class WtEngine core
    class WtRiskMonWrapper,WtRiskMonitor,IRiskMonitorFact wrapper
```

### 风控监视器包装类 WtRiskMonWrapper
**成员**：
- `WtRiskMonitor* _mon`：风控监视器指针
- `IRiskMonitorFact* _fact`：风控监视器工厂指针

参考 [Includes/note.ipynb/风险监控定义RiskMonDefs.h](../Includes/note.ipynb/风险监控定义RiskMonDefs.h)
```mermaid
graph TD
    subgraph "风险监控层 RiskMonDefs.h"
        style IRiskMonitorFact fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
        style WtRiskMonitor fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
        style WtPortContext fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
        
        IRiskMonitorFact["<b>IRiskMonitorFact (风控模块工厂接口)</b><br/><i>职责: 创建和销毁 *风控监控器*"]
        WtRiskMonitor["<b>WtRiskMonitor (风控监控器基类)</b><br/><i>职责: 封装具体的风控规则"]
        WtPortContext["<b>WtPortContext (风控模块上下文工具接口)</b><br/><i>职责: 提供监控所需的数据和执行控制的权限"]
    end

    %% --- 关系与数据流 (Relationships & Data Flow) ---

    %% 1. 创建与配置流程 (Creation & Configuration Flow)
    IRiskMonitorFact -->|"创建"| WtRiskMonitor

    %% 3. 依赖注入与核心交互循环 (Dependency Injection & Core Interaction Loop)
    WtRiskMonitor -->|"持有并使用"| WtPortContext
```

### 引擎事件监听器接口 IEngineEvtListener
定义引擎事件的监听接口，子类可以实现这些接口来监听引擎的各种事件。
- 初始化事件回调
    ```cpp
    /**
     * @brief 初始化事件回调
     * 当引擎初始化完成时被调用。
     */
    virtual void on_initialize_event() {}
    ```
- 定时事件回调
    ```cpp
    /**
     * @brief 定时事件回调
     * @param uDate 日期（格式：YYYYMMDD）
     * @param uTime 时间（格式：HHMM）
     * 当引擎定时触发时被调用。
     */
    virtual void on_schedule_event(uint32_t uDate, uint32_t uTime) {}
    ```
- 会话事件回调
    ```cpp
    /**
     * @brief 会话事件回调
     * @param uDate 日期（格式：YYYYMMDD）
     * @param isBegin 是否为会话开始，true表示开始，false表示结束
     * 当交易会话开始或结束时被调用。
     */
    virtual void on_session_event(uint32_t uDate, bool isBegin = true) {}
    ```

### 交易引擎基类 WtEngine
```cpp
class WtEngine : public WtPortContext, public IParserStub
```
- **WtPortContext**：参考 [Includes/note.ipynb/风险监控定义RiskMonDefs.h](../Includes/note.ipynb/风险监控定义RiskMonDefs.h)
    ```mermaid
    graph TD
        subgraph "风险监控层 RiskMonDefs.h"
            style IRiskMonitorFact fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
            style WtRiskMonitor fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
            style WtPortContext fill:#D2E0FB,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
            
            IRiskMonitorFact["<b>IRiskMonitorFact (风控模块工厂接口)</b><br/><i>职责: 创建和销毁 *风控监控器*"]
            WtRiskMonitor["<b>WtRiskMonitor (风控监控器基类)</b><br/><i>职责: 封装具体的风控规则"]
            WtPortContext["<b>WtPortContext (风控模块上下文工具接口)</b><br/><i>职责: 提供监控所需的数据和执行控制的权限"]
        end

        %% --- 关系与数据流 (Relationships & Data Flow) ---

        %% 1. 创建与配置流程 (Creation & Configuration Flow)
        IRiskMonitorFact -->|"创建"| WtRiskMonitor

        %% 3. 依赖注入与核心交互循环 (Dependency Injection & Core Interaction Loop)
        WtRiskMonitor -->|"持有并使用"| WtPortContext
    ```
- **IParserStub**：参考 [/适配器层/解析器适配器 ParserAdapter.h/cpp](./note.ipynb)

#### 成员
- **时间管理**
  - `uint32_t _cur_date`：当前日期（格式：YYYYMMDD）
  - `uint32_t _cur_time`：1分钟K线时间，指向下一根K线的时间（格式：HHMM），便于CTA策略使用
  - `uint32_t _cur_raw_time`：当前真实时间（格式：HHMM）
  - `uint32_t _cur_secs`：当前秒数（包含毫秒，格式：SSmmm）
  - `uint32_t _cur_tdate`：当前交易日（格式：YYYYMMDD）

- **核心管理器指针**
  - `IBaseDataMgr* _base_data_mgr`：基础数据管理器指针，用于获取合约信息等
  - `IHotMgr* _hot_mgr`：主力管理器指针，用于获取主力合约信息等
  - `WtDtMgr* _data_mgr`：数据管理器指针，用于获取行情数据等
  - `IEngineEvtListener* _evt_listener`：事件监听器指针，用于监听引擎事件
  - `TraderAdapterMgr* _adapter_mgr`：交易适配器管理器指针，用于访问交易接口

- **订阅管理**
  - `StraSubMap _tick_sub_map`：Tick数据订阅表，记录每个合约被哪些策略订阅了
  - `StraSubMap _bar_sub_map`：K线数据订阅表，记录每个合约被哪些策略订阅了
    - typedef wt_hashmap\<std::string, `SubList`\> StraSubMap
    - typedef wt_hashmap\<uint32_t, `SubOpt`\> SubList
    - typedef std::pair\<uint32_t, uint32_t\> SubOptpair
    - 本质上是 {合约代码：{策略ID：pair<策略ID，订阅选项>}}
      - 订阅选项：0=原始订阅，1=前复权，2=后复权
      ```JSON
      {
        // 合约1：SHFE.ag.1912
        "SHFE.ag.1912" -> {
            1001 -> pair(1001, 0)  // 策略1，原始订阅
        },
        
        // 合约2：SSE.600000（注意：键是去掉复权后缀的）
        "SSE.600000" -> {
            1002 -> pair(1002, 1),  // 策略2，前复权订阅
            1003 -> pair(1003, 2)   // 策略3，后复权订阅
        }
      }
      ```

- **信号管理**
  - `SignalMap _sig_map`：信号映射表，存储待触发的交易信号
    - typedef wt_hashmap\<std::string, `SigInfo`\> SignalMap：合约代码 ——> 信号信息
      ```cpp
      /* 信号信息结构体，用于存储待触发的交易信号信息 */
      typedef struct _SigInfo
      {
        double _volume; // 目标仓位数量
        uint64_t _gentime; // 信号生成时间戳
      }SigInfo;
      ```

- **过滤器与通知**
  - `WtFilterMgr _filter_mgr`：信号过滤器管理器，用于过滤或调整交易信号
  - `EventNotifier* _notifier`：事件通知器指针，用于发送事件通知

- **手续费模板**
  - `FeeMap _fee_map`：手续费映射表，存储各品种的手续费配置
    - typedef wt_hashmap\<std::string, `FeeItem`\> FeeMap：品种ID ——> 手续费项
      ```cpp
      /* 手续费项结构体，用于存储单个品种的手续费配置 */
      typedef struct _FeeItem
      {
        double _open; // 开仓手续费（按手数或按金额）
        double _close; // 平仓手续费（按手数或按金额）
        double _close_today; // 平今手续费（按手数或按金额）
        bool _by_volume; // 是否按手数计费，true表示按手数，false表示按金额
      } FeeItem;
      ```

- **资金管理**
  - `WTSPortFundInfo* _port_fund`：组合资金信息指针，存储组合的资金、持仓、盈亏等信息
    - 参考 [Includes/note.ipynb/交易层/交易统计信息和投资组合资金信息 WTSRiskDef.hpp/投资组合资金信息类 WTSPortFundInfo](../Includes/note.ipynb/交易层/交易统计信息和投资组合资金信息WTSRiskDef.hpp/投资组合资金信息类WTSPortFundInfo)
  - `uint32_t _fund_udt_span`：组合资金更新时间间隔（秒），0表示不限制更新间隔

- **持仓管理**
  - `PositionMap _pos_map`：持仓映射表，存储各合约的持仓信息
    - typedef wt_hashmap\<std::string, `PosInfoPtr`\> PositionMap：合约代码 ——> 持仓信息指针
    - typedef std::shared_ptr\<`PosInfo`\> PosInfoPtr：持仓信息智能指针类型定义
      ```cpp
      /* 持仓信息结构体 */
      typedef struct _PosInfo
      {
        double _volume; // 总持仓数量（正数表示多仓，负数表示空仓）
        double _closeprofit; // 累计平仓盈亏
        double _dynprofit; // 浮动盈亏
        SpinMutex	_mtx; // 自旋锁，用于保护持仓数据的线程安全
        std::vector<DetailInfo> _details;  // 持仓明细列表，存储每笔开仓的详细信息
      } PosInfo;

      /* 持仓明细信息结构体，用于存储单笔开仓的详细信息 */
      typedef struct _DetailInfo
      {
        bool _long; // 是否多仓，true表示多仓，false表示空仓
        double _price; // 开仓价格
        double _volume; // 持仓数量（大于0）
        uint64_t _opentime; // 开仓时间戳
        uint32_t _opentdate; // 开仓交易日（格式：YYYYMMDD）
        double _profit; // 浮动盈亏
      } DetailInfo;
      ```

- **价格缓存**
  - `PriceMap _price_map`：价格映射表，缓存各合约的最新价格
    - typedef wt_hashmap\<std::string, double\> PriceMap：合约代码 ——> 价格

- **后台任务线程**
  - `StdThreadPtr _thrd_task`：后台任务线程指针
  - `TaskQueue _task_queue`：任务队列，存储待处理的任务
    - typedef std::queue\<`TaskItem`\> TaskQueue：任务队列类型定义，存储任务项（函数对象）
    - typedef std::function\<void()\> TaskItem：任务项类型定义，无参数无返回值的函数对象
  - `StdUniqueMutex _mtx_task`：任务队列互斥锁，用于保护任务队列的线程安全
  - `StdCondVariable _cond_task`：任务队列条件变量，用于线程间通信
  - `bool _terminated`：终止标志，true表示线程已终止

- **风控管理**
  - `RiskMonFactInfo _risk_fact`：风控监视器工厂信息
      ```cpp
      /*风控监视器工厂信息结构体，用于存储风控模块的动态库信息和工厂指针。*/
      typedef struct _RiskMonFactInfo
      {
        std::string _module_path; // 风控模块路径（动态库文件路径）
        DllHandle _module_inst; // 动态库句柄
        IRiskMonitorFact*	_fact; // 风控监视器工厂指针
        FuncCreateRiskMonFact	_creator; // 创建工厂函数指针
        FuncDeleteRiskMonFact	_remover; // 删除工厂函数指针
      } RiskMonFactInfo;
      ```
  - `WtRiskMonPtr _risk_mon`：风控监视器智能指针
    - typedef std::shared_ptr\<`WtRiskMonWrapper`\> WtRiskMonPtr：风控监视器智能指针类型定义
  - `double _risk_volscale`：风控仓位缩放系数
  - `uint32_t _risk_date`：风控参数生效日期（格式：YYYYMMDD）

- **日志文件**
  - `BoostFilePtr _trade_logs`：成交记录文件指针
  - `BoostFilePtr _close_logs`：平仓记录文件指针

- **复权因子缓存**
  - `wt_hashmap<std::string, double> _factors_cache`：复权因子缓存映射表，键为合约代码，值为复权因子

- **状态标志**
  - `bool _ready`：就绪标志，true表示引擎已就绪，可以推送Tick数据

#### 方法

##### 初始化

###### 初始化引擎 init

##### 访问行情数据

###### 获取最后一个Tick数据 get_last_tick
```cpp
/**
 * @brief 获取最后一个Tick数据
 * @param sid 策略ID（未使用）
 * @param stdCode 标准合约代码字符串
 * @return WTSTickData* 返回最后一个Tick数据指针，如果不存在返回NULL
 */
WTSTickData* WtEngine::get_last_tick(uint32_t sid, const char* stdCode)
{
    // 从数据管理器获取最后一个Tick数据
	return _data_mgr->grab_last_tick(stdCode);
}
```

###### 获取Tick数据切片 get_tick_slice
```cpp
/**
 * @brief 获取Tick数据切片
 * @param sid 策略ID（未使用）
 * @param code 合约代码字符串
 * @param count 获取的Tick数量
 * @return WTSTickSlice* 返回Tick数据切片指针，如果不存在返回NULL
 */
WTSTickSlice* WtEngine::get_tick_slice(uint32_t sid, const char* code, uint32_t count)
{
    // 从数据管理器获取Tick数据切片
	return _data_mgr->get_tick_slice(code, count);
}
```

###### 获取K线数据切片 get_kline_slice
- 在 `_bar_sub_map[{stdCode}.{period}.{times}]` 加入 pair\<sid, 0\>
  - pair的第一个元素是策略ID，第二个元素是标志，0表示未复权
- 确定K线周期 kp（KP_Minute1/KP_Minute5/KP_DAY）和倍数
- 数据管理器调用并返回 `_data_mgr->get_kline_slice(stdCode, kp, times, count, etime)`
```cpp
/**
 * @brief 获取K线数据切片
 * @param sid 策略ID
 * @param stdCode 标准合约代码字符串
 * @param period 周期字符串（如"m1"表示1分钟，"d"表示日线）
 * @param count 获取的K线数量
 * @param times 周期倍数，默认为1（如period="m1"，times=5表示5分钟K线）
 * @param etime 结束时间（时间戳），默认为0（表示获取最新数据）
 * @return WTSKlineSlice* 返回K线数据切片指针，如果不存在返回NULL
 */
WTSKlineSlice* WtEngine::get_kline_slice(uint32_t sid, const char* stdCode, const char* period, uint32_t count, uint32_t times /* = 1 */, uint64_t etime /* = 0 */)
```

###### 订阅Tick数据 sub_tick

##### 价格管理

###### 获取当前价格 get_cur_price

###### 获取当日价格 get_day_price

###### 获取复权因子 get_exright_factor

###### 获取复权标志 get_adjusting_flag

##### 手续费计算

###### 计算手续费 calc_fee

###### 加载手续费模板 load_fees

##### WtPortContext 接口实现

###### 获取组合资金信息 getFundInfo

###### 设置仓位缩放系数 setVolScale

###### 判断是否在交易中 isInTrading

###### 写入风控日志 writeRiskLog

###### 获取当前日期 getCurDate

###### 获取当前时间 getCurTime

###### 获取交易日 getTradingDate

##### IParserStub 接口实现

###### 处理推送的行情数据 handle_push_quote

##### 生命周期与事件回调

###### 运行引擎 run

###### Tick事件处理 on_tick

###### K线事件处理 on_bar

###### 初始化事件处理 on_init

###### 交易会话开始事件处理 on_session_begin

###### 交易会话结束事件处理 on_session_end

##### 信号处理

###### 追加交易信号 append_signal

##### 持仓管理

###### 设置持仓 do_set_position

##### 资金管理

###### 更新资金浮动盈亏 update_fund_dynprofit

##### 任务调度

###### 任务循环 task_loop

###### 推送任务 push_task

##### 数据持久化

###### 加载数据 load_datas

###### 保存数据 save_datas

##### 日志记录 

###### 记录成交日志 log_trade

###### 记录平仓日志 log_close

## WtCtaEngine.h/cpp

## WtHftEngine.h/cpp

## WtSelEngine.h/cpp

# Ticker层

## WtCtaTicker.h/cpp

## WtHftTicker.h/cpp

## WtSelTicker.h/cpp

# 执行器层

## WtExecuterFactory.h/cpp

## WtLocalExecuter.h/cpp

## WtArbiExecuter.h/cpp

## WtDiffExecuter.h/cpp

## WtDistExecuter.h/cpp

## WtExecMgr.h/cpp

# 策略上下文和管理层

## CtaStraBaseCtx.h/cpp

## HftStraBaseCtx.h/cpp

## SelStraBaseCtx.h/cpp

## CtaStraContext.h/cpp

## HftStraContext.h/cpp

## SelStraContext.h/cpp

## CtaStrategyMgr.h/cpp

## HftStrategyMgr.h/cpp

## SelStrategyMgr.h/cpp